# 109 — CTRP and PRISM Acquisition and Audit

## Objective

Audit and freeze the CTRP and PRISM pharmacogenomic resources required for
Phase 6 — Pharmacogenomic Contexts and Explainable Modeling.

The primary CTRP source is the original CTD²-distributed
`CTRPv2.0_2015_ctd2_ExpandedDataset.zip`. The separately acquired
`PSet_CTRPv2.rds` is retained as a secondary PharmacoGx/ORCESTRA-derived
resource for provenance and potential cross-checking, but it is not treated as
the primary CTRP pharmacological input.

The PRISM source is the PRISM Repurposing 19Q4 Secondary Screen, represented
by the acquired dose-response, cell-line, treatment-metadata, documentation,
and replicate-collapsed response files.

This notebook will establish byte-level identity and source provenance for the
acquired files, inspect their documented structure, and characterize the
identifier and response-metadata systems required for the Phase 6 prerequisite
audit.

## Main tasks

This notebook will:

1. verify the presence, size, and SHA-256 identity of the acquired CTRP and
   PRISM files;
2. document the canonical source, release, acquisition role, and provenance of
   each resource;
3. inspect the CTRP archive contents without modifying the original ZIP;
4. inspect the structural schemas and documentation of the acquired
   pharmacogenomic resources;
5. characterize the available cell-line, compound, experiment, treatment, and
   response identifiers needed for deterministic downstream harmonization;
6. distinguish primary source files from supporting or derived resources; and
7. provide the audited raw-data handoff required before the Phase 6
   metadata-only prerequisite assessment.

## Analytical boundary

This notebook is an acquisition and infrastructure audit.

It will not:

- test consensus-program–drug associations;
- select compounds based on pharmacological results;
- define Phase 6 drug-eligibility or coverage thresholds;
- choose a primary response metric based on observed associations;
- fit predictive models;
- calculate SHAP or other model attributions;
- perform cross-screen replication;
- use Phase 5 vulnerability evidence to prioritize compounds;
- alter the frozen Phase 3–5 analytical artifacts; or
- interpret pharmacological response as clinical resistance.

Response values may be inspected only to the extent required to verify file
structure, metric availability, orientation documentation, missingness, or
technical integrity. Any Phase 6 analytical choices that could depend on
biological or association results must remain prospective.

## Downstream handoff

The audited resources from notebook 109 will support a Phase 6 metadata-only
prerequisite audit of screen structure, identifier compatibility, coverage,
cell-line overlap, compound mapping, and consensus-program score portability.

Only after that prerequisite audit will the Phase 6 analysis contract be
finalized. Program–drug association testing and predictive modeling will remain
outside notebook 109.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import hashlib
import zipfile
import json

import pandas as pd
import numpy as np
from copy import deepcopy

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import (
    validate_raw_data_registry,
)
from pancancer_epigenetics.utils.raw_data_registry import (
    load_raw_data_registry,
    validate_raw_data_registry,
)

In [2]:
# =============================================================================
# CTRP and PRISM input paths
# =============================================================================

CTRP_ARCHIVE_PATH = (
    Paths.ctrp
    / "CTRPv2.0_2015_ctd2_ExpandedDataset.zip"
)

CTRP_PHARMACOGX_PATH = (
    Paths.ctrp
    / "PSet_CTRPv2.rds"
)

PRISM_README_PATH = (
    Paths.prism
    / "secondary-screen-readme.txt"
)

PRISM_CELL_LINE_INFO_PATH = (
    Paths.prism
    / "secondary-screen-cell-line-info.csv"
)

PRISM_DOSE_RESPONSE_PATH = (
    Paths.prism
    / "secondary-screen-dose-response-curve-parameters.csv"
)

PRISM_TREATMENT_INFO_PATH = (
    Paths.prism
    / "secondary-screen-replicate-collapsed-treatment-info.csv"
)

PRISM_LOGFOLD_CHANGE_PATH = (
    Paths.prism
    / "secondary-screen-replicate-collapsed-logfold-change.csv"
)

In [3]:
# =============================================================================
# Validate acquired CTRP and PRISM inputs
# =============================================================================

pharmacogenomic_input_paths = {
    "ctrp_archive": CTRP_ARCHIVE_PATH,
    "ctrp_pharmacogx": CTRP_PHARMACOGX_PATH,
    "prism_readme": PRISM_README_PATH,
    "prism_cell_line_info": PRISM_CELL_LINE_INFO_PATH,
    "prism_dose_response": PRISM_DOSE_RESPONSE_PATH,
    "prism_treatment_info": PRISM_TREATMENT_INFO_PATH,
    "prism_logfold_change": PRISM_LOGFOLD_CHANGE_PATH,
}

missing_inputs = [
    name
    for name, path in pharmacogenomic_input_paths.items()
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing acquired pharmacogenomic inputs: "
        + ", ".join(missing_inputs)
    )

for name, path in pharmacogenomic_input_paths.items():
    print(f"{name}: {project_relative_path(path)}")

ctrp_archive: data/raw/ctrp/CTRPv2.0_2015_ctd2_ExpandedDataset.zip
ctrp_pharmacogx: data/raw/ctrp/PSet_CTRPv2.rds
prism_readme: data/raw/prism/secondary-screen-readme.txt
prism_cell_line_info: data/raw/prism/secondary-screen-cell-line-info.csv
prism_dose_response: data/raw/prism/secondary-screen-dose-response-curve-parameters.csv
prism_treatment_info: data/raw/prism/secondary-screen-replicate-collapsed-treatment-info.csv
prism_logfold_change: data/raw/prism/secondary-screen-replicate-collapsed-logfold-change.csv


In [4]:
# =============================================================================
# Byte-level inventory of acquired pharmacogenomic files
# =============================================================================

pharmacogenomic_file_inventory = pd.DataFrame(
    [
        {
            "resource": resource,
            "relative_path": project_relative_path(path),
            "size_bytes": path.stat().st_size,
            "sha256": calculate_sha256(path),
        }
        for resource, path in pharmacogenomic_input_paths.items()
    ]
)

pharmacogenomic_file_inventory

,resource,relative_path,size_bytes,sha256
0,ctrp_archive,data/raw/ctrp/CTRPv2.0_2015_ctd2_ExpandedDatas...,342737645,8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301...
1,ctrp_pharmacogx,data/raw/ctrp/PSet_CTRPv2.rds,40707609,95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade2...
2,prism_readme,data/raw/prism/secondary-screen-readme.txt,7647,ed453e2bbaecff26cf923d8a69290f2c18fd7488d5a7d6...
3,prism_cell_line_info,data/raw/prism/secondary-screen-cell-line-info...,40979,b93436b5f4bcf5fd14589697be4ca8f99ffd99d6392230...
4,prism_dose_response,data/raw/prism/secondary-screen-dose-response-...,264340589,88d1013506e0cd6f191a51c5f3fdd3fb2be54f8afb4e19...
5,prism_treatment_info,data/raw/prism/secondary-screen-replicate-coll...,3788355,9d0d1fb4faa87a63cd84965ec5e2b55a9df5680520a41b...
6,prism_logfold_change,data/raw/prism/secondary-screen-replicate-coll...,91620551,a358beb9efbc96b3d777cbb0212e4cd724080f17970d24...


In [5]:
# =============================================================================
# Report complete byte-level identities
# =============================================================================

for row in pharmacogenomic_file_inventory.itertuples(index=False):
    print(f"{row.resource}")
    print(f"  path:   {row.relative_path}")
    print(f"  bytes:  {row.size_bytes}")
    print(f"  sha256: {row.sha256}")

ctrp_archive
  path:   data/raw/ctrp/CTRPv2.0_2015_ctd2_ExpandedDataset.zip
  bytes:  342737645
  sha256: 8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301474b707648689bdee3
ctrp_pharmacogx
  path:   data/raw/ctrp/PSet_CTRPv2.rds
  bytes:  40707609
  sha256: 95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade282a5e4e45200ee442e
prism_readme
  path:   data/raw/prism/secondary-screen-readme.txt
  bytes:  7647
  sha256: ed453e2bbaecff26cf923d8a69290f2c18fd7488d5a7d6903821980e6d49c99d
prism_cell_line_info
  path:   data/raw/prism/secondary-screen-cell-line-info.csv
  bytes:  40979
  sha256: b93436b5f4bcf5fd14589697be4ca8f99ffd99d639223080380bc514c3718d27
prism_dose_response
  path:   data/raw/prism/secondary-screen-dose-response-curve-parameters.csv
  bytes:  264340589
  sha256: 88d1013506e0cd6f191a51c5f3fdd3fb2be54f8afb4e19a5d1f8538e81fbfec8
prism_treatment_info
  path:   data/raw/prism/secondary-screen-replicate-collapsed-treatment-info.csv
  bytes:  3788355
  sha256: 9d0d1fb4faa87a63cd84965ec5e2

In [6]:
# =============================================================================
# Cross-check against independently observed file identities
# =============================================================================

expected_file_identities = {
    "ctrp_archive": {
        "size_bytes": 342737645,
        "sha256": "8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301474b707648689bdee3",
    },
    "ctrp_pharmacogx": {
        "size_bytes": 40707609,
        "sha256": "95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade282a5e4e45200ee442e",
    },
    "prism_readme": {
        "size_bytes": 7647,
        "sha256": "ed453e2bbaecff26cf923d8a69290f2c18fd7488d5a7d6903821980e6d49c99d",
    },
    "prism_cell_line_info": {
        "size_bytes": 40979,
        "sha256": "b93436b5f4bcf5fd14589697be4ca8f99ffd99d639223080380bc514c3718d27",
    },
    "prism_dose_response": {
        "size_bytes": 264340589,
        "sha256": "88d1013506e0cd6f191a51c5f3fdd3fb2be54f8afb4e19a5d1f8538e81fbfec8",
    },
    "prism_treatment_info": {
        "size_bytes": 3788355,
        "sha256": "9d0d1fb4faa87a63cd84965ec5e2b55a9df5680520a41b029abb679cfd4384f7",
    },
    "prism_logfold_change": {
        "size_bytes": 91620551,
        "sha256": "a358beb9efbc96b3d777cbb0212e4cd724080f17970d241627ccd17855d21939",
    },
}

for row in pharmacogenomic_file_inventory.itertuples(index=False):
    expected = expected_file_identities[row.resource]

    assert row.size_bytes == expected["size_bytes"], (
        f"Size mismatch for {row.resource}"
    )
    assert row.sha256 == expected["sha256"], (
        f"SHA256 mismatch for {row.resource}"
    )

print("All acquired CTRP and PRISM files match the independently observed identities.")

All acquired CTRP and PRISM files match the independently observed identities.


In [7]:
# =============================================================================
# Validate CTRP archive integrity
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    bad_member = ctrp_zip.testzip()
    member_count = len(ctrp_zip.infolist())
    total_uncompressed_bytes = sum(
        member.file_size
        for member in ctrp_zip.infolist()
    )

if bad_member is not None:
    raise RuntimeError(
        f"CTRP archive CRC validation failed for: {bad_member}"
    )

print("CTRP archive CRC validation: PASS")
print(f"Archive members: {member_count:,}")
print(f"Total uncompressed bytes: {total_uncompressed_bytes:,}")

CTRP archive CRC validation: PASS
Archive members: 16
Total uncompressed bytes: 1,929,832,496


In [8]:
# =============================================================================
# Inventory CTRP archive members
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    ctrp_archive_inventory = pd.DataFrame(
        [
            {
                "file_name": member.filename,
                "uncompressed_bytes": member.file_size,
                "compressed_bytes": member.compress_size,
            }
            for member in ctrp_zip.infolist()
        ]
    )

ctrp_archive_inventory

,file_name,uncompressed_bytes,compressed_bytes
0,CTRPv2.0._COLUMNS.xlsx,16625,13912
1,CTRPv2.0._INFORMER_SET.xlsx,78383,75616
2,CTRPv2.0._README.docx,13313,10526
3,MANIFEST.txt,886,483
4,v20._COLUMNS.txt,9480,2235
5,v20.data.curves_post_qc.txt,59801043,17931598
6,v20.data.per_cpd_avg.txt,409162056,67446317
7,v20.data.per_cpd_post_qc.txt,331401478,63333543
8,v20.data.per_cpd_pre_qc.txt,331270238,41553546
9,v20.data.per_cpd_well.txt,797366581,152249928


In [9]:
print(ctrp_archive_inventory)

                       file_name  uncompressed_bytes  compressed_bytes
0         CTRPv2.0._COLUMNS.xlsx               16625             13912
1    CTRPv2.0._INFORMER_SET.xlsx               78383             75616
2          CTRPv2.0._README.docx               13313             10526
3                   MANIFEST.txt                 886               483
4               v20._COLUMNS.txt                9480              2235
5    v20.data.curves_post_qc.txt            59801043          17931598
6       v20.data.per_cpd_avg.txt           409162056          67446317
7   v20.data.per_cpd_post_qc.txt           331401478          63333543
8    v20.data.per_cpd_pre_qc.txt           331270238          41553546
9      v20.data.per_cpd_well.txt           797366581         152249928
10       v20.meta.media_comp.txt                4272              1096
11  v20.meta.per_assay_plate.txt              422350             62389
12    v20.meta.per_cell_line.txt               71591             10810
13    

In [10]:
# =============================================================================
# Validate expected CTRPv2.0 archive contents
# =============================================================================

expected_ctrp_members = {
    "CTRPv2.0._COLUMNS.xlsx",
    "CTRPv2.0._INFORMER_SET.xlsx",
    "CTRPv2.0._README.docx",
    "MANIFEST.txt",
    "v20._COLUMNS.txt",
    "v20.data.curves_post_qc.txt",
    "v20.data.per_cpd_avg.txt",
    "v20.data.per_cpd_post_qc.txt",
    "v20.data.per_cpd_pre_qc.txt",
    "v20.data.per_cpd_well.txt",
    "v20.meta.media_comp.txt",
    "v20.meta.per_assay_plate.txt",
    "v20.meta.per_cell_line.txt",
    "v20.meta.per_compound.txt",
    "v20.meta.per_experiment.txt",
    "v20._README.txt",
}

observed_ctrp_members = set(ctrp_archive_inventory["file_name"])

missing_members = expected_ctrp_members - observed_ctrp_members
unexpected_members = observed_ctrp_members - expected_ctrp_members

if missing_members or unexpected_members:
    raise RuntimeError(
        "Unexpected CTRPv2.0 archive contents.\n"
        f"Missing: {sorted(missing_members)}\n"
        f"Unexpected: {sorted(unexpected_members)}"
    )

print("CTRPv2.0 archive contents: PASS")

CTRPv2.0 archive contents: PASS


In [11]:
# =============================================================================
# Validate CTRP members against the provider manifest
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    manifest_text = ctrp_zip.read("MANIFEST.txt").decode("utf-8")

    manifest_records = []
    for line in manifest_text.splitlines():
        line = line.strip()
        if not line:
            continue

        expected_md5, file_name = line.split(maxsplit=1)
        file_name = file_name.lstrip("*")

        md5 = hashlib.md5()
        with ctrp_zip.open(file_name) as handle:
            while chunk := handle.read(1024 * 1024):
                md5.update(chunk)

        manifest_records.append(
            {
                "file_name": file_name,
                "expected_md5": expected_md5,
                "observed_md5": md5.hexdigest(),
            }
        )

ctrp_manifest_audit = pd.DataFrame(manifest_records)
ctrp_manifest_audit["md5_match"] = (
    ctrp_manifest_audit["expected_md5"]
    == ctrp_manifest_audit["observed_md5"]
)

if not ctrp_manifest_audit["md5_match"].all():
    failed_files = ctrp_manifest_audit.loc[
        ~ctrp_manifest_audit["md5_match"],
        "file_name",
    ].tolist()
    raise RuntimeError(
        "CTRP provider-manifest validation failed for: "
        + ", ".join(failed_files)
    )

print(
    "CTRP provider-manifest MD5 validation: PASS "
    f"({len(ctrp_manifest_audit):,} files)"
)

CTRP provider-manifest MD5 validation: PASS (15 files)


In [12]:
# =============================================================================
# Inspect CTRPv2.0 provider README
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    ctrp_readme_text = (
        ctrp_zip.read("v20._README.txt")
        .decode("utf-8")
    )

print(ctrp_readme_text)

Small-Molecule Cancer Cell-Line Sensitivity Profiling Data

CTRPv2.0_2015_ctd2_ExpandedDataset.zip

The package contains 10 data files, a README file that describes the data, and a COLUMNS file of descriptions of column headers shared by the 10 data files.

metadata files
v20.meta.per_compound.txt: contextual compound information and annotation
v20.meta.per_cell_line.txt: contextual cancer cell line information and annotation
v20.meta.per_experiment.txt: information about experimental growth conditions, media, and SNP fingerprinting
v20.meta.per_assay_plate.txt: statistical information about DMSO vehicle-control distributions on each assay plate
v20.meta.media_comp.txt: basal media names and short description of media additives

data files
v20.data.per_cpd_well.txt: raw and transformed viability values for each cancer cell line treated with compound for each concentration point tested for each replicate tested
v20.data.per_cpd_avg.txt: transformed and averaged viability values and erro

In [13]:
# =============================================================================
# Inspect CTRPv2.0 column dictionary
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    ctrp_columns_text = (
        ctrp_zip.read("v20._COLUMNS.txt")
        .decode("utf-8")
    )

print(ctrp_columns_text)

DATA_FILENAME	COLUMN_HEADER	COLUMN_DESCRIPTION
v20.data.curves_post_qc.txt	experiment_id	serial number for experiment performed during CTRPv2 data collection
v20.data.curves_post_qc.txt	conc_pts_fit	number of concentration-response points participating in curve fit
v20.data.curves_post_qc.txt	fit_num_param	number of free parameters in curve fit
v20.data.curves_post_qc.txt	p1_conf_int_high	"upper bound of confidence interval (95%) for curve fit parameter 1, center: log2(apparent_ec50_umol)"
v20.data.curves_post_qc.txt	p1_conf_int_low	"lower bound of confidence interval (95%) for curve fit parameter 1, center: log2(apparent_ec50_umol)"
v20.data.curves_post_qc.txt	p2_conf_int_high	"upper bound of confidence interval (95%) for curve fit parameter 2, slope parameter"
v20.data.curves_post_qc.txt	p2_conf_int_low	"lower bound of confidence interval (95%) for curve fit parameter 2, slope parameter"
v20.data.curves_post_qc.txt	p4_conf_int_high	"upper bound of confidence interval (95%) for curve 

In [14]:
# =============================================================================
# Load key CTRPv2.0 metadata and response schema
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    with ctrp_zip.open("v20.meta.per_cell_line.txt") as handle:
        ctrp_cell_lines = pd.read_csv(handle, sep="\t")

    with ctrp_zip.open("v20.meta.per_compound.txt") as handle:
        ctrp_compounds = pd.read_csv(handle, sep="\t")

    with ctrp_zip.open("v20.meta.per_experiment.txt") as handle:
        ctrp_experiments = pd.read_csv(handle, sep="\t")

    with ctrp_zip.open("v20.data.curves_post_qc.txt") as handle:
        ctrp_curves_schema = pd.read_csv(
            handle,
            sep="\t",
            nrows=0,
        )

for name, table in {
    "cell_lines": ctrp_cell_lines,
    "compounds": ctrp_compounds,
    "experiments": ctrp_experiments,
}.items():
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]} columns")

print()
print("curves_post_qc columns:")
print(ctrp_curves_schema.columns.tolist())

cell_lines: 1,107 rows x 6 columns
compounds: 545 rows x 11 columns
experiments: 1,061 rows x 9 columns

curves_post_qc columns:
['experiment_id', 'conc_pts_fit', 'fit_num_param', 'p1_conf_int_high', 'p1_conf_int_low', 'p2_conf_int_high', 'p2_conf_int_low', 'p4_conf_int_high', 'p4_conf_int_low', 'p1_center', 'p2_slope', 'p3_total_decline', 'p4_baseline', 'apparent_ec50_umol', 'pred_pv_high_conc', 'area_under_curve', 'master_cpd_id']


In [15]:
# =============================================================================
# Audit CTRPv2.0 metadata key integrity
# =============================================================================

ctrp_key_audit = pd.DataFrame(
    [
        {
            "table": "cell_lines",
            "key": "master_ccl_id",
            "rows": len(ctrp_cell_lines),
            "unique_keys": ctrp_cell_lines["master_ccl_id"].nunique(dropna=False),
            "duplicate_keys": ctrp_cell_lines["master_ccl_id"].duplicated().sum(),
            "missing_keys": ctrp_cell_lines["master_ccl_id"].isna().sum(),
        },
        {
            "table": "compounds",
            "key": "master_cpd_id",
            "rows": len(ctrp_compounds),
            "unique_keys": ctrp_compounds["master_cpd_id"].nunique(dropna=False),
            "duplicate_keys": ctrp_compounds["master_cpd_id"].duplicated().sum(),
            "missing_keys": ctrp_compounds["master_cpd_id"].isna().sum(),
        },
        {
            "table": "experiments",
            "key": "experiment_id",
            "rows": len(ctrp_experiments),
            "unique_keys": ctrp_experiments["experiment_id"].nunique(dropna=False),
            "duplicate_keys": ctrp_experiments["experiment_id"].duplicated().sum(),
            "missing_keys": ctrp_experiments["experiment_id"].isna().sum(),
        },
    ]
)

ctrp_key_audit

,table,key,rows,unique_keys,duplicate_keys,missing_keys
0,cell_lines,master_ccl_id,1107,1107,0,0
1,compounds,master_cpd_id,545,545,0,0
2,experiments,experiment_id,1061,907,154,0


In [16]:
orphan_experiment_cell_lines = (
    set(ctrp_experiments["master_ccl_id"].dropna())
    - set(ctrp_cell_lines["master_ccl_id"].dropna())
)

print(
    "Experiment → cell-line orphan references:",
    len(orphan_experiment_cell_lines),
)

Experiment → cell-line orphan references: 0


In [17]:
# =============================================================================
# Characterize duplicated CTRPv2.0 experiment identifiers
# =============================================================================

duplicate_experiment_rows = ctrp_experiments.loc[
    ctrp_experiments["experiment_id"].duplicated(keep=False)
].copy()

experiment_duplicate_summary = (
    duplicate_experiment_rows
    .groupby("experiment_id", dropna=False)
    .agg(
        rows=("experiment_id", "size"),
        unique_cell_lines=("master_ccl_id", "nunique"),
        unique_runs=("run_id", "nunique"),
        unique_dates=("experiment_date", "nunique"),
        unique_media=("culture_media", "nunique"),
    )
    .reset_index()
)

print(
    "Experiment IDs represented more than once:",
    experiment_duplicate_summary["experiment_id"].nunique(),
)
print(
    "Exact duplicate rows:",
    ctrp_experiments.duplicated().sum(),
)
print(
    "Duplicated experiment IDs mapping to >1 cell line:",
    (experiment_duplicate_summary["unique_cell_lines"] > 1).sum(),
)

experiment_duplicate_summary.head(20)

Experiment IDs represented more than once: 153
Exact duplicate rows: 0
Duplicated experiment IDs mapping to >1 cell line: 0


,experiment_id,rows,unique_cell_lines,unique_runs,unique_dates,unique_media
0,84,2,1,1,2,1
1,85,2,1,1,2,1
2,86,2,1,1,2,1
3,87,2,1,1,2,1
4,88,2,1,1,2,1
5,89,2,1,1,2,1
6,90,2,1,1,2,1
7,91,2,1,1,2,1
8,92,2,1,1,2,1
9,93,2,1,1,2,1


In [18]:
# =============================================================================
# Identify metadata variation within duplicated experiment identifiers
# =============================================================================

duplicate_group_sizes = (
    duplicate_experiment_rows
    .groupby("experiment_id")
    .size()
    .value_counts()
    .sort_index()
)

variation_by_column = {}

metadata_columns = [
    column
    for column in ctrp_experiments.columns
    if column != "experiment_id"
]

for column in metadata_columns:
    n_varying_experiments = (
        duplicate_experiment_rows
        .groupby("experiment_id")[column]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )

    variation_by_column[column] = n_varying_experiments

print("Rows per duplicated experiment_id:")
print(duplicate_group_sizes)

print("\nNumber of duplicated experiment_ids with variation by column:")
for column, count in variation_by_column.items():
    print(f"{column}: {count}")

Rows per duplicated experiment_id:
2    152
3      1
Name: count, dtype: int64

Number of duplicated experiment_ids with variation by column:
run_id: 1
experiment_date: 153
culture_media: 0
baseline_signal: 0
cells_per_well: 0
growth_mode: 0
snp_fp_status: 0
master_ccl_id: 0


In [19]:
# =============================================================================
# Inspect the experiment identifier with multiple run IDs
# =============================================================================

multi_run_experiment_ids = (
    duplicate_experiment_rows
    .groupby("experiment_id")["run_id"]
    .nunique(dropna=False)
    .loc[lambda x: x > 1]
    .index
)

multi_run_experiment_rows = (
    ctrp_experiments
    .loc[
        ctrp_experiments["experiment_id"].isin(
            multi_run_experiment_ids
        )
    ]
    .sort_values(["experiment_id", "experiment_date", "run_id"])
)

multi_run_experiment_rows

,experiment_id,run_id,experiment_date,culture_media,baseline_signal,cells_per_well,growth_mode,snp_fp_status,master_ccl_id
257,234,7024-01-A01-02-22,20120723,RPMI001,0.1195,500,adherent,SNP-matched-reference,1191
258,234,7024-01-A01-02-22,20120724,RPMI001,0.1195,500,adherent,SNP-matched-reference,1191
259,234,7024-01-A01-02-23,20120727,RPMI001,0.1195,500,adherent,SNP-matched-reference,1191


In [20]:
# =============================================================================
# Build deterministic CTRPv2.0 experiment-to-cell-line mapping
# =============================================================================

ctrp_experiment_cell_map = (
    ctrp_experiments[
        ["experiment_id", "master_ccl_id"]
    ]
    .drop_duplicates()
    .sort_values("experiment_id")
    .reset_index(drop=True)
)

if ctrp_experiment_cell_map["experiment_id"].duplicated().any():
    raise RuntimeError(
        "experiment_id does not map uniquely to master_ccl_id."
    )

if ctrp_experiment_cell_map["master_ccl_id"].isna().any():
    raise RuntimeError(
        "Missing master_ccl_id in experiment-to-cell-line mapping."
    )

print(
    "Unique experiment IDs:",
    f"{len(ctrp_experiment_cell_map):,}",
)
print(
    "Mapped cell lines:",
    f"{ctrp_experiment_cell_map['master_ccl_id'].nunique():,}",
)
print("Experiment → cell-line mapping: PASS")

Unique experiment IDs: 907
Mapped cell lines: 887
Experiment → cell-line mapping: PASS


In [21]:
# =============================================================================
# Audit CTRPv2.0 response-table identifiers
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    with ctrp_zip.open("v20.data.curves_post_qc.txt") as handle:
        ctrp_curve_ids = pd.read_csv(
            handle,
            sep="\t",
            usecols=[
                "experiment_id",
                "master_cpd_id",
            ],
        )

duplicate_response_pairs = ctrp_curve_ids.duplicated(
    subset=["experiment_id", "master_cpd_id"]
).sum()

orphan_curve_experiments = (
    set(ctrp_curve_ids["experiment_id"].dropna())
    - set(ctrp_experiment_cell_map["experiment_id"])
)

orphan_curve_compounds = (
    set(ctrp_curve_ids["master_cpd_id"].dropna())
    - set(ctrp_compounds["master_cpd_id"])
)

print(f"Response rows: {len(ctrp_curve_ids):,}")
print(
    "Unique experiment IDs:",
    f"{ctrp_curve_ids['experiment_id'].nunique():,}",
)
print(
    "Unique compound IDs:",
    f"{ctrp_curve_ids['master_cpd_id'].nunique():,}",
)
print(
    "Duplicated experiment-compound pairs:",
    f"{duplicate_response_pairs:,}",
)
print(
    "Orphan experiment references:",
    f"{len(orphan_curve_experiments):,}",
)
print(
    "Orphan compound references:",
    f"{len(orphan_curve_compounds):,}",
)

Response rows: 395,263
Unique experiment IDs: 907
Unique compound IDs: 545
Duplicated experiment-compound pairs: 0
Orphan experiment references: 0
Orphan compound references: 0


In [22]:
# =============================================================================
# Audit CTRPv2.0 response-metric completeness and ranges
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    with ctrp_zip.open("v20.data.curves_post_qc.txt") as handle:
        ctrp_response_metrics = pd.read_csv(
            handle,
            sep="\t",
            usecols=[
                "experiment_id",
                "master_cpd_id",
                "conc_pts_fit",
                "fit_num_param",
                "apparent_ec50_umol",
                "pred_pv_high_conc",
                "area_under_curve",
            ],
        )

metric_columns = [
    "conc_pts_fit",
    "fit_num_param",
    "apparent_ec50_umol",
    "pred_pv_high_conc",
    "area_under_curve",
]

ctrp_metric_audit = pd.DataFrame(
    [
        {
            "metric": column,
            "dtype": str(ctrp_response_metrics[column].dtype),
            "missing_n": ctrp_response_metrics[column].isna().sum(),
            "missing_pct": (
                ctrp_response_metrics[column].isna().mean() * 100
            ),
            "min": ctrp_response_metrics[column].min(),
            "median": ctrp_response_metrics[column].median(),
            "max": ctrp_response_metrics[column].max(),
        }
        for column in metric_columns
    ]
)

ctrp_metric_audit

,metric,dtype,missing_n,missing_pct,min,median,max
0,conc_pts_fit,int64,0,0.0,8.0000,16.0000,2.900000e+01
1,fit_num_param,int64,0,0.0,2.0000,3.0000,3.000000e+00
2,apparent_ec50_umol,float64,0,0.0,0.0000,3.4510,2.274000e+307
3,pred_pv_high_conc,float64,0,0.0,-0.0997,0.4561,2.658900e+00
4,area_under_curve,float64,0,0.0,0.0691,13.4490,2.935000e+01


In [23]:
# =============================================================================
# Characterize extreme CTRPv2.0 fitted EC50 values
# =============================================================================

with zipfile.ZipFile(CTRP_ARCHIVE_PATH, mode="r") as ctrp_zip:
    with ctrp_zip.open("v20.data.curves_post_qc.txt") as handle:
        ctrp_ec50_audit = pd.read_csv(
            handle,
            sep="\t",
            usecols=[
                "experiment_id",
                "master_cpd_id",
                "p1_center",
                "apparent_ec50_umol",
            ],
        )

ec50_quantiles = (
    ctrp_ec50_audit["apparent_ec50_umol"]
    .quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
    .rename("apparent_ec50_umol")
)

print("Apparent EC50 quantiles (µM):")
print(ec50_quantiles)

print("\nExtreme apparent EC50 counts:")
for threshold in [10, 100, 1_000, 1e6, 1e12, 1e100]:
    count = (
        ctrp_ec50_audit["apparent_ec50_umol"] > threshold
    ).sum()
    print(
        f"> {threshold:g} µM: "
        f"{count:,} "
        f"({count / len(ctrp_ec50_audit) * 100:.4f}%)"
    )

print(
    "\nMaximum p1_center:",
    ctrp_ec50_audit["p1_center"].max(),
)

Apparent EC50 quantiles (µM):
0.000     0.000000e+00
0.500     3.451000e+00
0.900     1.064000e+02
0.950     1.219000e+03
0.990     2.593180e+13
0.999     7.506036e+95
1.000    2.274000e+307
Name: apparent_ec50_umol, dtype: float64

Extreme apparent EC50 counts:
> 10 µM: 142,007 (35.9272%)
> 100 µM: 40,644 (10.2828%)
> 1000 µM: 20,291 (5.1335%)
> 1e+06 µM: 11,577 (2.9289%)
> 1e+12 µM: 4,683 (1.1848%)
> 1e+100 µM: 379 (0.0959%)

Maximum p1_center: 2.237e+20


In [24]:
# =============================================================================
# Compare fitted EC50 values with compound test concentrations
# =============================================================================

ctrp_ec50_vs_test_range = (
    ctrp_ec50_audit
    .merge(
        ctrp_compounds[
            ["master_cpd_id", "top_test_conc_umol"]
        ],
        on="master_cpd_id",
        how="left",
        validate="many_to_one",
    )
)

if ctrp_ec50_vs_test_range["top_test_conc_umol"].isna().any():
    raise RuntimeError(
        "Missing top_test_conc_umol after compound metadata join."
    )

ctrp_ec50_vs_test_range["ec50_to_top_conc_ratio"] = (
    ctrp_ec50_vs_test_range["apparent_ec50_umol"]
    / ctrp_ec50_vs_test_range["top_test_conc_umol"]
)

print(
    "Rows with apparent EC50 above top test concentration:",
    f"{(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 1).sum():,}",
    f"({(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 1).mean() * 100:.2f}%)",
)

print(
    "Rows with apparent EC50 >10x top test concentration:",
    f"{(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 10).sum():,}",
    f"({(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 10).mean() * 100:.2f}%)",
)

print(
    "Rows with apparent EC50 >100x top test concentration:",
    f"{(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 100).sum():,}",
    f"({(ctrp_ec50_vs_test_range['ec50_to_top_conc_ratio'] > 100).mean() * 100:.2f}%)",
)

print("\nEC50 / top-test-concentration ratio quantiles:")
print(
    ctrp_ec50_vs_test_range["ec50_to_top_conc_ratio"]
    .quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
)

Rows with apparent EC50 above top test concentration: 39,905 (10.10%)
Rows with apparent EC50 >10x top test concentration: 21,032 (5.32%)
Rows with apparent EC50 >100x top test concentration: 16,683 (4.22%)

EC50 / top-test-concentration ratio quantiles:
0.000     0.000000e+00
0.500     6.384848e-02
0.900     1.004383e+00
0.950     1.634296e+01
0.990     3.908212e+11
0.999     1.034534e+94
1.000    6.890909e+305
Name: ec50_to_top_conc_ratio, dtype: float64


In [25]:
# =============================================================================
# Audit CTRPv2.0 cell-line–compound response multiplicity
# =============================================================================

ctrp_cell_compound_pairs = (
    ctrp_curve_ids
    .merge(
        ctrp_experiment_cell_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )
)

cell_compound_multiplicity = (
    ctrp_cell_compound_pairs
    .groupby(
        ["master_ccl_id", "master_cpd_id"],
        dropna=False,
    )
    .size()
)

print(
    "Unique cell-line–compound pairs:",
    f"{len(cell_compound_multiplicity):,}",
)
print(
    "Pairs represented by >1 experiment:",
    f"{(cell_compound_multiplicity > 1).sum():,}",
    f"({(cell_compound_multiplicity > 1).mean() * 100:.2f}%)",
)
print(
    "Maximum experiments per cell-line–compound pair:",
    f"{cell_compound_multiplicity.max():,}",
)

print("\nMultiplicity distribution:")
print(
    cell_compound_multiplicity
    .value_counts()
    .sort_index()
)

Unique cell-line–compound pairs: 387,130
Pairs represented by >1 experiment: 7,708 (1.99%)
Maximum experiments per cell-line–compound pair: 3

Multiplicity distribution:
1    379422
2      7283
3       425
Name: count, dtype: int64


In [26]:
# =============================================================================
# Inspect PRISM provider README
# =============================================================================

prism_readme_text = PRISM_README_PATH.read_text(
    encoding="utf-8"
)

print(prism_readme_text)

prism_repurposing_secondary

The secondary PRISM Repurposing dataset contains the results of pooled-cell line chemical-perturbation viability screens for 1,448 compounds screened against 489 cell lines in an 8-step, 4-fold dilution, starting from 10uM. Technical redos for 147 previously screened oncology compounds have been added as of 11/22/2019. This data is identified using a `screen_id` of "MTS010".  It is recommended to use MTS010 when available. MTS010 was not included in the analysis for Corsello et al. 2019, DOI:10.1101/730119.


Data are processed according to the following protocol:

- Calculate the median fluorescence intensity (MFI) of each bead-replicate pair.
- Remove outlier-pools. MFI values are log-transformed and centered to the median logMFI for each cell line on each detection plate. For each well on a plate, the median of the centered values is standardized (using median and MAD) against all other wells in the same position in the same screen. Data from wells with 

In [27]:
# =============================================================================
# Load PRISM metadata and inspect response schemas
# =============================================================================

prism_cell_lines = pd.read_csv(
    PRISM_CELL_LINE_INFO_PATH
)

prism_treatments = pd.read_csv(
    PRISM_TREATMENT_INFO_PATH
)

prism_dose_response_schema = pd.read_csv(
    PRISM_DOSE_RESPONSE_PATH,
    nrows=0,
)

prism_logfold_change_schema = pd.read_csv(
    PRISM_LOGFOLD_CHANGE_PATH,
    nrows=0,
)

print(
    "cell_line_info:",
    f"{prism_cell_lines.shape[0]:,} rows x "
    f"{prism_cell_lines.shape[1]} columns",
)

print(
    "replicate_collapsed_treatment_info:",
    f"{prism_treatments.shape[0]:,} rows x "
    f"{prism_treatments.shape[1]} columns",
)

print("\ndose_response columns:")
print(prism_dose_response_schema.columns.tolist())

print("\nreplicate_collapsed_logfold_change:")
print(
    f"{len(prism_logfold_change_schema.columns):,} columns"
)
print(
    "First 10 columns:",
    prism_logfold_change_schema.columns[:10].tolist(),
)

cell_line_info: 588 rows x 7 columns
replicate_collapsed_treatment_info: 13,008 rows x 12 columns

dose_response columns:
['broad_id', 'depmap_id', 'ccle_name', 'screen_id', 'upper_limit', 'lower_limit', 'slope', 'r2', 'auc', 'ec50', 'ic50', 'name', 'moa', 'target', 'disease.area', 'indication', 'smiles', 'phase', 'passed_str_profiling', 'row_name']

replicate_collapsed_logfold_change:
13,009 columns
First 10 columns: ['Unnamed: 0', 'BRD-A00077618-236-07-6::0.00061034::HTS002', 'BRD-A00077618-236-07-6::0.0024414::HTS002', 'BRD-A00077618-236-07-6::0.00976562::HTS002', 'BRD-A00077618-236-07-6::0.0390625::HTS002', 'BRD-A00077618-236-07-6::0.15625::HTS002', 'BRD-A00077618-236-07-6::0.625::HTS002', 'BRD-A00077618-236-07-6::10::HTS002', 'BRD-A00077618-236-07-6::2.5::HTS002', 'BRD-A00758722-001-04-9::0.00061034::HTS002']


In [28]:
# =============================================================================
# Audit PRISM metadata identifiers and matrix-column alignment
# =============================================================================

print("cell_line_info columns:")
print(prism_cell_lines.columns.tolist())

print("\nreplicate_collapsed_treatment_info columns:")
print(prism_treatments.columns.tolist())

print("\nCell-line metadata:")
print(
    "rows:",
    f"{len(prism_cell_lines):,}",
)
print(
    "unique row_name:",
    f"{prism_cell_lines['row_name'].nunique(dropna=False):,}",
)
print(
    "duplicate row_name:",
    f"{prism_cell_lines['row_name'].duplicated().sum():,}",
)
print(
    "missing depmap_id:",
    f"{prism_cell_lines['depmap_id'].isna().sum():,}",
)
print(
    "missing ccle_name:",
    f"{prism_cell_lines['ccle_name'].isna().sum():,}",
)

matrix_condition_columns = set(
    prism_logfold_change_schema.columns[1:]
)
treatment_condition_columns = set(
    prism_treatments["column_name"]
)

print("\nTreatment ↔ collapsed-matrix alignment:")
print(
    "treatment rows:",
    f"{len(prism_treatments):,}",
)
print(
    "unique treatment column_name:",
    f"{prism_treatments['column_name'].nunique(dropna=False):,}",
)
print(
    "matrix condition columns:",
    f"{len(matrix_condition_columns):,}",
)
print(
    "treatments absent from matrix:",
    f"{len(treatment_condition_columns - matrix_condition_columns):,}",
)
print(
    "matrix columns absent from treatment metadata:",
    f"{len(matrix_condition_columns - treatment_condition_columns):,}",
)

cell_line_info columns:
['row_name', 'depmap_id', 'ccle_name', 'primary_tissue', 'secondary_tissue', 'tertiary_tissue', 'passed_str_profiling']

replicate_collapsed_treatment_info columns:
['column_name', 'broad_id', 'dose', 'screen_id', 'compound_plate', 'name', 'moa', 'target', 'disease.area', 'indication', 'smiles', 'phase']

Cell-line metadata:
rows: 588
unique row_name: 588
duplicate row_name: 0
missing depmap_id: 20
missing ccle_name: 20

Treatment ↔ collapsed-matrix alignment:
treatment rows: 13,008
unique treatment column_name: 13,008
matrix condition columns: 13,008
treatments absent from matrix: 0
matrix columns absent from treatment metadata: 0


In [29]:
# =============================================================================
# Audit PRISM cell-line participation across response resources
# =============================================================================

prism_dose_response_ids = pd.read_csv(
    PRISM_DOSE_RESPONSE_PATH,
    usecols=[
        "row_name",
        "depmap_id",
        "ccle_name",
        "screen_id",
        "passed_str_profiling",
    ],
)

prism_logfold_change_rows = pd.read_csv(
    PRISM_LOGFOLD_CHANGE_PATH,
    usecols=[0],
).iloc[:, 0]

dose_row_names = set(
    prism_dose_response_ids["row_name"].dropna()
)

lfc_row_names = set(
    prism_logfold_change_rows.dropna()
)

metadata_row_names = set(
    prism_cell_lines["row_name"].dropna()
)

print("Dose-response:")
print(
    "rows:",
    f"{len(prism_dose_response_ids):,}",
)
print(
    "unique row_name:",
    f"{prism_dose_response_ids['row_name'].nunique():,}",
)
print(
    "unique depmap_id:",
    f"{prism_dose_response_ids['depmap_id'].nunique():,}",
)
print(
    "unique ccle_name:",
    f"{prism_dose_response_ids['ccle_name'].nunique():,}",
)

print("\nUnique cell lines by screen_id:")
print(
    prism_dose_response_ids
    .groupby("screen_id")["row_name"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nReplicate-collapsed log-fold-change matrix:")
print(
    "rows:",
    f"{len(prism_logfold_change_rows):,}",
)
print(
    "unique row IDs:",
    f"{len(lfc_row_names):,}",
)

print("\nCross-resource row-name alignment:")
print(
    "Dose-response rows absent from cell_line_info:",
    f"{len(dose_row_names - metadata_row_names):,}",
)
print(
    "LFC rows absent from cell_line_info:",
    f"{len(lfc_row_names - metadata_row_names):,}",
)
print(
    "Dose-response rows absent from LFC matrix:",
    f"{len(dose_row_names - lfc_row_names):,}",
)
print(
    "LFC rows absent from dose-response:",
    f"{len(lfc_row_names - dose_row_names):,}",
)
print(
    "Metadata rows unused by either response resource:",
    f"{len(metadata_row_names - (dose_row_names | lfc_row_names)):,}",
)

Dose-response:
rows: 701,004
unique row_name: 488
unique depmap_id: 480
unique ccle_name: 481

Unique cell lines by screen_id:
screen_id
HTS002    488
MTS006    484
MTS010    480
MTS005    451
Name: row_name, dtype: int64

Replicate-collapsed log-fold-change matrix:
rows: 489
unique row IDs: 489

Cross-resource row-name alignment:
Dose-response rows absent from cell_line_info: 8
LFC rows absent from cell_line_info: 8
Dose-response rows absent from LFC matrix: 0
LFC rows absent from dose-response: 1
Metadata rows unused by either response resource: 107


In [30]:
# =============================================================================
# Characterize PRISM cross-resource cell-line discrepancies
# =============================================================================

response_rows_missing_metadata = sorted(
    (dose_row_names | lfc_row_names) - metadata_row_names
)

lfc_only_rows = sorted(
    lfc_row_names - dose_row_names
)

discrepant_dose_metadata = (
    prism_dose_response_ids
    .loc[
        prism_dose_response_ids["row_name"].isin(
            response_rows_missing_metadata
        )
    ]
    .groupby("row_name", dropna=False)
    .agg(
        depmap_ids=("depmap_id", lambda x: sorted(set(x.dropna()))),
        ccle_names=("ccle_name", lambda x: sorted(set(x.dropna()))),
        screen_ids=("screen_id", lambda x: sorted(set(x.dropna()))),
        str_values=(
            "passed_str_profiling",
            lambda x: sorted(set(x.dropna().astype(str))),
        ),
        response_rows=("row_name", "size"),
    )
    .reset_index()
)

print("Response row_names absent from cell_line_info:")
print(response_rows_missing_metadata)

print("\nLFC row_names absent from dose-response:")
print(lfc_only_rows)

print("\nAvailable dose-response metadata for missing cell-line-info rows:")
discrepant_dose_metadata

Response row_names absent from cell_line_info:
['ACH-000010_FAILED_STR', 'ACH-000028_FAILED_STR', 'ACH-000409_FAILED_STR', 'ACH-000511_FAILED_STR', 'ACH-000539_FAILED_STR', 'ACH-000807_FAILED_STR', 'ACH-000925_FAILED_STR', 'ACH-001078_FAILED_STR']

LFC row_names absent from dose-response:
['ACH-001192']

Available dose-response metadata for missing cell-line-info rows:


,row_name,depmap_ids,ccle_names,screen_ids,str_values,response_rows
0,ACH-000010_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1301
1,ACH-000028_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1336
2,ACH-000409_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1459
3,ACH-000511_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1379
4,ACH-000539_FAILED_STR,[],[],"[HTS002, MTS006]",[False],838
5,ACH-000807_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1530
6,ACH-000925_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1556
7,ACH-001078_FAILED_STR,[],[],"[HTS002, MTS005, MTS006, MTS010]",[False],1413


In [31]:
# =============================================================================
# Resolve PRISM failed-STR row names and inspect LFC-only cell line
# =============================================================================

failed_str_resolution = pd.DataFrame(
    {
        "response_row_name": response_rows_missing_metadata
    }
)

failed_str_resolution["base_row_name"] = (
    failed_str_resolution["response_row_name"]
    .str.replace("_FAILED_STR", "", regex=False)
)

failed_str_resolution["base_present_in_cell_line_info"] = (
    failed_str_resolution["base_row_name"]
    .isin(metadata_row_names)
)

failed_str_resolution = (
    failed_str_resolution
    .merge(
        prism_cell_lines,
        left_on="base_row_name",
        right_on="row_name",
        how="left",
        suffixes=("", "_metadata"),
    )
)

print("FAILED_STR response rows:")
display(
    failed_str_resolution[
        [
            "response_row_name",
            "base_row_name",
            "base_present_in_cell_line_info",
            "depmap_id",
            "ccle_name",
            "passed_str_profiling",
        ]
    ]
)

print("\nLFC-only row metadata:")
display(
    prism_cell_lines.loc[
        prism_cell_lines["row_name"].isin(lfc_only_rows)
    ]
)

FAILED_STR response rows:


,response_row_name,base_row_name,base_present_in_cell_line_info,depmap_id,ccle_name,passed_str_profiling
0,ACH-000010_FAILED_STR,ACH-000010,True,NaN,NaN,False
1,ACH-000028_FAILED_STR,ACH-000028,True,NaN,NaN,False
2,ACH-000409_FAILED_STR,ACH-000409,True,NaN,NaN,False
3,ACH-000511_FAILED_STR,ACH-000511,True,NaN,NaN,False
4,ACH-000539_FAILED_STR,ACH-000539,True,NaN,NaN,False
5,ACH-000807_FAILED_STR,ACH-000807,True,NaN,NaN,False
6,ACH-000925_FAILED_STR,ACH-000925,True,NaN,NaN,False
7,ACH-001078_FAILED_STR,ACH-001078,True,NaN,NaN,False



LFC-only row metadata:


,row_name,depmap_id,ccle_name,primary_tissue,secondary_tissue,tertiary_tissue,passed_str_profiling
445,ACH-001192,ACH-001192,SKNEP1_BONE,bone,Ewing_sarcoma,NaN,True


In [32]:
# =============================================================================
# Report failed-STR row-name resolution
# =============================================================================

print(
    failed_str_resolution[
        [
            "response_row_name",
            "base_row_name",
            "base_present_in_cell_line_info",
            "depmap_id",
            "ccle_name",
            "passed_str_profiling",
        ]
    ].to_string(index=False)
)

    response_row_name base_row_name  base_present_in_cell_line_info depmap_id ccle_name passed_str_profiling
ACH-000010_FAILED_STR    ACH-000010                            True       NaN       NaN                False
ACH-000028_FAILED_STR    ACH-000028                            True       NaN       NaN                False
ACH-000409_FAILED_STR    ACH-000409                            True       NaN       NaN                False
ACH-000511_FAILED_STR    ACH-000511                            True       NaN       NaN                False
ACH-000539_FAILED_STR    ACH-000539                            True       NaN       NaN                False
ACH-000807_FAILED_STR    ACH-000807                            True       NaN       NaN                False
ACH-000925_FAILED_STR    ACH-000925                            True       NaN       NaN                False
ACH-001078_FAILED_STR    ACH-001078                            True       NaN       NaN                False


In [33]:
# =============================================================================
# Inspect cell-line metadata for failed-STR base identifiers
# =============================================================================

failed_str_base_rows = (
    prism_cell_lines
    .loc[
        prism_cell_lines["row_name"].isin(
            failed_str_resolution["base_row_name"]
        )
    ]
    .sort_values("row_name")
)

print(
    failed_str_base_rows.to_string(index=False)
)

  row_name depmap_id ccle_name         primary_tissue     secondary_tissue         tertiary_tissue passed_str_profiling
ACH-000010       NaN       NaN                   lung             lung_NSC     lung_adenocarcinoma                False
ACH-000028       NaN       NaN                 breast                ERpos                     NaN                False
ACH-000409       NaN       NaN                  ovary ovary_adenocarcinoma ovary_high_grade_serous                False
ACH-000511       NaN       NaN                   lung             lung_NSC           lung_squamous                False
ACH-000539       NaN       NaN             fibroblast                  NaN                     NaN                False
ACH-000807       NaN       NaN central_nervous_system               glioma       oligodendroglioma                False
ACH-000925       NaN       NaN                   lung             lung_NSC                     NaN                False
ACH-001078       NaN       NaN          

In [34]:
# =============================================================================
# Validate PRISM cell-line identity consistency
# =============================================================================

dose_identity_audit = (
    prism_dose_response_ids
    .groupby("row_name", dropna=False)
    .agg(
        unique_depmap_ids=("depmap_id", lambda x: x.dropna().nunique()),
        unique_ccle_names=("ccle_name", lambda x: x.dropna().nunique()),
        unique_str_values=("passed_str_profiling", lambda x: x.nunique(dropna=False)),
    )
    .reset_index()
)

print(
    "Dose-response row_names with >1 depmap_id:",
    f"{(dose_identity_audit['unique_depmap_ids'] > 1).sum():,}",
)
print(
    "Dose-response row_names with >1 ccle_name:",
    f"{(dose_identity_audit['unique_ccle_names'] > 1).sum():,}",
)
print(
    "Dose-response row_names with inconsistent STR status:",
    f"{(dose_identity_audit['unique_str_values'] > 1).sum():,}",
)

dose_identity_lookup = (
    prism_dose_response_ids[
        ["row_name", "depmap_id", "ccle_name"]
    ]
    .drop_duplicates()
)

metadata_identity_lookup = prism_cell_lines[
    ["row_name", "depmap_id", "ccle_name"]
]

identity_comparison = (
    dose_identity_lookup
    .merge(
        metadata_identity_lookup,
        on="row_name",
        how="left",
        suffixes=("_dose", "_metadata"),
        validate="many_to_one",
    )
)

valid_identity_rows = identity_comparison[
    identity_comparison["depmap_id_dose"].notna()
].copy()

depmap_mismatch = (
    valid_identity_rows["depmap_id_dose"]
    != valid_identity_rows["depmap_id_metadata"]
)

ccle_mismatch = (
    valid_identity_rows["ccle_name_dose"]
    != valid_identity_rows["ccle_name_metadata"]
)

print(
    "\nMapped dose-response row_names:",
    f"{valid_identity_rows['row_name'].nunique():,}",
)
print(
    "DepMap ID mismatches versus cell_line_info:",
    f"{depmap_mismatch.sum():,}",
)
print(
    "CCLE name mismatches versus cell_line_info:",
    f"{ccle_mismatch.sum():,}",
)

Dose-response row_names with >1 depmap_id: 0
Dose-response row_names with >1 ccle_name: 1
Dose-response row_names with inconsistent STR status: 0

Mapped dose-response row_names: 480
DepMap ID mismatches versus cell_line_info: 0
CCLE name mismatches versus cell_line_info: 1


In [35]:
# =============================================================================
# Inspect the PRISM CCLE-name inconsistency
# =============================================================================

ccle_inconsistent_rows = (
    dose_identity_audit
    .loc[
        dose_identity_audit["unique_ccle_names"] > 1,
        "row_name",
    ]
    .tolist()
)

print("row_name with multiple CCLE names:")
print(ccle_inconsistent_rows)

print("\nDose-response records:")
print(
    prism_dose_response_ids
    .loc[
        prism_dose_response_ids["row_name"].isin(
            ccle_inconsistent_rows
        ),
        [
            "row_name",
            "depmap_id",
            "ccle_name",
            "screen_id",
            "passed_str_profiling",
        ],
    ]
    .drop_duplicates()
    .sort_values(["row_name", "screen_id", "ccle_name"])
    .to_string(index=False)
)

print("\ncell_line_info metadata:")
print(
    prism_cell_lines
    .loc[
        prism_cell_lines["row_name"].isin(
            ccle_inconsistent_rows
        )
    ]
    .to_string(index=False)
)

row_name with multiple CCLE names:
['ACH-000037']

Dose-response records:
  row_name  depmap_id        ccle_name screen_id  passed_str_profiling
ACH-000037 ACH-000037 S117_SOFT_TISSUE    HTS002                  True
ACH-000037 ACH-000037 S117_SOFT_TISSUE    MTS005                  True
ACH-000037 ACH-000037 S117_SOFT_TISSUE    MTS006                  True
ACH-000037 ACH-000037     S117_THYROID    MTS010                  True

cell_line_info metadata:
  row_name  depmap_id    ccle_name primary_tissue secondary_tissue tertiary_tissue passed_str_profiling
ACH-000037 ACH-000037 S117_THYROID        thyroid  thyroid_sarcoma             NaN                 True


In [36]:
# =============================================================================
# Characterize PRISM secondary-screen structure
# =============================================================================

prism_response_screen_summary = (
    prism_dose_response_ids
    .groupby("screen_id")
    .agg(
        response_rows=("row_name", "size"),
        unique_row_names=("row_name", "nunique"),
        unique_depmap_ids=("depmap_id", "nunique"),
    )
)

prism_treatment_screen_summary = (
    prism_treatments
    .groupby("screen_id")
    .agg(
        treatment_conditions=("column_name", "size"),
        unique_broad_ids=("broad_id", "nunique"),
        unique_compound_names=("name", "nunique"),
        unique_doses=("dose", "nunique"),
    )
)

prism_screen_summary = (
    prism_response_screen_summary
    .join(
        prism_treatment_screen_summary,
        how="outer",
    )
    .sort_index()
)

prism_screen_summary

,response_rows,unique_row_names,unique_depmap_ids,treatment_conditions,unique_broad_ids,unique_compound_names,unique_doses
screen_id,,,,,,,
HTS002,602495,488,480,11168,1396,1394,933
MTS005,834,451,444,16,2,2,10
MTS006,33472,484,476,584,73,73,51
MTS010,64203,480,473,1240,147,147,81


In [37]:
# =============================================================================
# Audit PRISM compound overlap across screens
# =============================================================================

compound_screen_presence = (
    prism_treatments[
        ["broad_id", "screen_id"]
    ]
    .drop_duplicates()
    .assign(present=True)
    .pivot(
        index="broad_id",
        columns="screen_id",
        values="present",
    )
    .fillna(False)
    .astype(bool)
)

screen_ids = sorted(
    prism_treatments["screen_id"].dropna().unique()
)

print("Unique compounds across all screens:")
print(f"{len(compound_screen_presence):,}")

print("\nPairwise compound overlap:")
for i, screen_a in enumerate(screen_ids):
    for screen_b in screen_ids[i + 1:]:
        overlap = (
            compound_screen_presence[screen_a]
            & compound_screen_presence[screen_b]
        ).sum()

        print(
            f"{screen_a} ∩ {screen_b}: "
            f"{overlap:,}"
        )

print("\nMTS010 compound disposition:")
mts010_compounds = set(
    prism_treatments.loc[
        prism_treatments["screen_id"] == "MTS010",
        "broad_id",
    ]
)

for screen_id in ["HTS002", "MTS005", "MTS006"]:
    screen_compounds = set(
        prism_treatments.loc[
            prism_treatments["screen_id"] == screen_id,
            "broad_id",
        ]
    )

    print(
        f"MTS010 compounds also present in {screen_id}: "
        f"{len(mts010_compounds & screen_compounds):,} / "
        f"{len(mts010_compounds):,}"
    )

Unique compounds across all screens:
1,502

Pairwise compound overlap:
HTS002 ∩ MTS005: 0
HTS002 ∩ MTS006: 20
HTS002 ∩ MTS010: 87
MTS005 ∩ MTS006: 0
MTS005 ∩ MTS010: 0
MTS006 ∩ MTS010: 10

MTS010 compound disposition:
MTS010 compounds also present in HTS002: 87 / 147
MTS010 compounds also present in MTS005: 0 / 147
MTS010 compounds also present in MTS006: 10 / 147


In [38]:
# =============================================================================
# Audit PRISM compound identity across drug-batch identifiers
# =============================================================================

prism_compound_identity = (
    prism_treatments[
        ["broad_id", "name"]
    ]
    .drop_duplicates()
)

broad_id_name_counts = (
    prism_compound_identity
    .groupby("broad_id")["name"]
    .nunique(dropna=False)
)

name_broad_id_counts = (
    prism_compound_identity
    .groupby("name")["broad_id"]
    .nunique(dropna=False)
)

print(
    "Unique broad_id values:",
    f"{prism_compound_identity['broad_id'].nunique():,}",
)
print(
    "Unique compound names:",
    f"{prism_compound_identity['name'].nunique():,}",
)
print(
    "broad_id values mapping to >1 name:",
    f"{(broad_id_name_counts > 1).sum():,}",
)
print(
    "Compound names mapping to >1 broad_id:",
    f"{(name_broad_id_counts > 1).sum():,}",
)

print("\nNumber of broad_id values per compound name:")
print(
    name_broad_id_counts
    .value_counts()
    .sort_index()
)

Unique broad_id values: 1,502
Unique compound names: 1,448
broad_id values mapping to >1 name: 0
Compound names mapping to >1 broad_id: 54

Number of broad_id values per compound name:
broad_id
1    1394
2      54
Name: count, dtype: int64


In [39]:
# =============================================================================
# Characterize PRISM compounds represented by multiple drug batches
# =============================================================================

multi_batch_names = (
    name_broad_id_counts
    .loc[lambda x: x > 1]
    .index
)

multi_batch_compounds = (
    prism_treatments
    .loc[
        prism_treatments["name"].isin(multi_batch_names),
        [
            "name",
            "broad_id",
            "screen_id",
            "smiles",
        ],
    ]
    .drop_duplicates()
)

multi_batch_summary = (
    multi_batch_compounds
    .groupby("name", dropna=False)
    .agg(
        broad_ids=(
            "broad_id",
            lambda x: sorted(set(x.dropna())),
        ),
        screens=(
            "screen_id",
            lambda x: sorted(set(x.dropna())),
        ),
        unique_smiles=(
            "smiles",
            lambda x: x.dropna().nunique(),
        ),
    )
    .reset_index()
)

print(
    "Multi-batch compound names:",
    f"{len(multi_batch_summary):,}",
)
print(
    "Names with >1 non-missing SMILES representation:",
    f"{(multi_batch_summary['unique_smiles'] > 1).sum():,}",
)

print("\nScreen-combination frequencies:")
print(
    multi_batch_summary["screens"]
    .astype(str)
    .value_counts()
)

multi_batch_summary.head(20)

Multi-batch compound names: 54
Names with >1 non-missing SMILES representation: 0

Screen-combination frequencies:
screens
['HTS002', 'MTS010']    49
['HTS002']               2
['MTS006', 'MTS010']     2
['HTS002', 'MTS005']     1
Name: count, dtype: int64


,name,broad_ids,screens,unique_smiles
0,3-amino-benzamide,"[BRD-K08703257-001-12-1, BRD-K08703257-001-13-9]","[HTS002, MTS010]",1
1,AZ-628,"[BRD-K05804044-001-06-0, BRD-K05804044-001-18-5]","[HTS002, MTS010]",1
2,BMS-690514,"[BRD-K76239644-001-01-8, BRD-K76239644-001-02-6]","[HTS002, MTS010]",1
3,BMS-754807,"[BRD-K13049116-001-04-0, BRD-K13049116-001-05-7]","[HTS002, MTS010]",1
4,NVP-BEZ235,"[BRD-K12184916-001-15-4, BRD-K12184916-001-19-6]","[HTS002, MTS010]",1
5,U-0126,"[BRD-K18787491-001-08-6, BRD-K91701654-001-03-1]",[HTS002],1
6,afatinib,"[BRD-K66175015-001-09-0, BRD-K66175015-001-12-4]","[HTS002, MTS010]",1
7,alectinib,"[BRD-K11267252-001-04-4, BRD-K11267252-001-05-1]","[HTS002, MTS010]",1
8,alvespimycin,"[BRD-K83988098-001-02-0, BRD-K83988098-003-03-4]","[HTS002, MTS010]",1
9,aminoglutethimide,"[BRD-A25234499-001-18-3, BRD-A25234499-001-19-1]","[HTS002, MTS010]",1


In [40]:
# =============================================================================
# Audit PRISM dose-response observation granularity
# =============================================================================

prism_dose_response_keys = pd.read_csv(
    PRISM_DOSE_RESPONSE_PATH,
    usecols=[
        "broad_id",
        "name",
        "row_name",
        "depmap_id",
        "screen_id",
    ],
)

batch_level_duplicates = (
    prism_dose_response_keys
    .duplicated(
        subset=[
            "broad_id",
            "row_name",
            "screen_id",
        ]
    )
    .sum()
)

name_level_multiplicity = (
    prism_dose_response_keys
    .groupby(
        [
            "name",
            "row_name",
            "screen_id",
        ],
        dropna=False,
    )["broad_id"]
    .nunique()
)

print(
    "Dose-response rows:",
    f"{len(prism_dose_response_keys):,}",
)

print(
    "Duplicated broad_id × row_name × screen_id observations:",
    f"{batch_level_duplicates:,}",
)

print(
    "Name × row_name × screen_id combinations represented by >1 broad_id:",
    f"{(name_level_multiplicity > 1).sum():,}",
)

print(
    "Maximum broad_id values per name × row_name × screen_id:",
    f"{name_level_multiplicity.max():,}",
)

print("\nMultiplicity distribution:")
print(
    name_level_multiplicity
    .value_counts()
    .sort_index()
)

Dose-response rows: 701,004
Duplicated broad_id × row_name × screen_id observations: 0
Name × row_name × screen_id combinations represented by >1 broad_id: 917
Maximum broad_id values per name × row_name × screen_id: 2

Multiplicity distribution:
broad_id
1    699170
2       917
Name: count, dtype: int64


In [41]:
# =============================================================================
# Characterize within-screen multi-batch PRISM observations
# =============================================================================

multi_batch_observations = (
    name_level_multiplicity
    .loc[lambda x: x > 1]
    .rename("n_broad_ids")
    .reset_index()
)

multi_batch_screen_summary = (
    multi_batch_observations
    .groupby("screen_id")
    .agg(
        multi_batch_cell_compound_pairs=("name", "size"),
        unique_compound_names=("name", "nunique"),
        unique_cell_lines=("row_name", "nunique"),
    )
    .sort_values(
        "multi_batch_cell_compound_pairs",
        ascending=False,
    )
)

print(
    "Multi-batch name × row_name × screen_id combinations:",
    f"{len(multi_batch_observations):,}",
)

print(
    "Compound names involved:",
    f"{multi_batch_observations['name'].nunique():,}",
)

print("\nDistribution by screen:")
display(multi_batch_screen_summary)

print("\nMost frequent compound names among multi-batch observations:")
print(
    multi_batch_observations["name"]
    .value_counts()
    .head(20)
)

Multi-batch name × row_name × screen_id combinations: 917
Compound names involved: 2

Distribution by screen:


,multi_batch_cell_compound_pairs,unique_compound_names,unique_cell_lines
screen_id,,,
HTS002,917,2,483



Most frequent compound names among multi-batch observations:
name
doxycycline    463
U-0126         454
Name: count, dtype: int64


In [42]:
# =============================================================================
# Inspect within-screen multi-batch PRISM compounds
# =============================================================================

within_screen_multi_batch_names = sorted(
    multi_batch_observations["name"].unique()
)

within_screen_batch_metadata = (
    prism_treatments
    .loc[
        (prism_treatments["screen_id"] == "HTS002")
        & prism_treatments["name"].isin(
            within_screen_multi_batch_names
        ),
        [
            "name",
            "broad_id",
            "screen_id",
            "dose",
            "smiles",
        ],
    ]
)

within_screen_batch_summary = (
    within_screen_batch_metadata
    .groupby(
        ["name", "broad_id", "screen_id"],
        dropna=False,
    )
    .agg(
        n_doses=("dose", "nunique"),
        min_dose_umol=("dose", "min"),
        max_dose_umol=("dose", "max"),
        unique_smiles=("smiles", "nunique"),
    )
    .reset_index()
    .sort_values(["name", "broad_id"])
)

print(within_screen_batch_summary.to_string(index=False))

print("\nDose grids:")
for (name, broad_id), group in (
    within_screen_batch_metadata
    .groupby(["name", "broad_id"])
):
    doses = sorted(group["dose"].dropna().unique())
    print(f"{name} | {broad_id}")
    print(doses)

       name               broad_id screen_id  n_doses  min_dose_umol  max_dose_umol  unique_smiles
     U-0126 BRD-K18787491-001-08-6    HTS002        8       0.000610        10.0000              1
     U-0126 BRD-K91701654-001-03-1    HTS002        8       0.000651        10.6695              1
doxycycline BRD-A08545410-003-07-8    HTS002        8       0.000580         9.5028              1
doxycycline BRD-A08545410-311-03-4    HTS002        8       0.000610        10.0000              1

Dose grids:
U-0126 | BRD-K18787491-001-08-6
[np.float64(0.00061034), np.float64(0.0024414), np.float64(0.00976562), np.float64(0.0390625), np.float64(0.15625), np.float64(0.625), np.float64(2.5), np.float64(10.0)]
U-0126 | BRD-K91701654-001-03-1
[np.float64(0.000651216), np.float64(0.00260487), np.float64(0.0104195), np.float64(0.0416778), np.float64(0.166711), np.float64(0.666846), np.float64(2.66738), np.float64(10.6695)]
doxycycline | BRD-A08545410-003-07-8
[np.float64(0.000580005), np.float64(0.

In [43]:
# =============================================================================
# Audit PRISM dose-response metric completeness and ranges
# =============================================================================

prism_response_metrics = pd.read_csv(
    PRISM_DOSE_RESPONSE_PATH,
    usecols=[
        "broad_id",
        "row_name",
        "screen_id",
        "upper_limit",
        "lower_limit",
        "slope",
        "r2",
        "auc",
        "ec50",
        "ic50",
    ],
)

prism_metric_columns = [
    "upper_limit",
    "lower_limit",
    "slope",
    "r2",
    "auc",
    "ec50",
    "ic50",
]

prism_metric_audit = pd.DataFrame(
    [
        {
            "metric": column,
            "dtype": str(prism_response_metrics[column].dtype),
            "missing_n": prism_response_metrics[column].isna().sum(),
            "missing_pct": (
                prism_response_metrics[column].isna().mean() * 100
            ),
            "min": prism_response_metrics[column].min(),
            "median": prism_response_metrics[column].median(),
            "max": prism_response_metrics[column].max(),
        }
        for column in prism_metric_columns
    ]
)

prism_metric_audit

,metric,dtype,missing_n,missing_pct,min,median,max
0,upper_limit,int64,0,0.000000,1.000000e+00,1.000000,1.000000e+00
1,lower_limit,float64,0,0.000000,-6.366571e+03,0.435089,5.023206e+03
2,slope,float64,0,0.000000,-4.891334e+03,1.511117,1.853737e+04
3,r2,float64,0,0.000000,-5.837855e+05,0.228972,1.000000e+00
4,auc,float64,0,0.000000,4.174068e-03,0.915402,4.889162e+00
5,ec50,float64,0,0.000000,1.160622e-07,0.487758,3.877887e+304
6,ic50,float64,339092,48.372335,0.000000e+00,1.499405,inf


In [44]:
# =============================================================================
# Characterize PRISM non-finite and extreme response metrics
# =============================================================================

for metric in ["r2", "auc", "ec50", "ic50"]:
    values = prism_response_metrics[metric]

    finite_mask = np.isfinite(values)
    positive_inf = np.isposinf(values).sum()
    negative_inf = np.isneginf(values).sum()

    print(f"{metric}")
    print(f"  finite: {finite_mask.sum():,}")
    print(f"  missing NaN: {values.isna().sum():,}")
    print(f"  +inf: {positive_inf:,}")
    print(f"  -inf: {negative_inf:,}")

    finite_values = values.loc[finite_mask]

    print("  finite quantiles:")
    print(
        finite_values
        .quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
        .to_string()
    )
    print()

r2
  finite: 701,004
  missing NaN: 0
  +inf: 0
  -inf: 0
  finite quantiles:
0.000   -583785.498385
0.500         0.228972
0.900         0.724796
0.950         0.809976
0.990         0.917354
0.999         0.990569
1.000         1.000000

auc
  finite: 701,004
  missing NaN: 0
  +inf: 0
  -inf: 0
  finite quantiles:
0.000    0.004174
0.500    0.915402
0.900    1.336905
0.950    1.454801
0.990    1.715837
0.999    2.134436
1.000    4.889162

ec50
  finite: 701,004
  missing NaN: 0
  +inf: 0
  -inf: 0
  finite quantiles:
0.000     1.160622e-07
0.500     4.877578e-01
0.900     5.209307e+00
0.950     8.774264e+00
0.990     2.051002e+08
0.999     7.833772e+75
1.000    3.877887e+304

ic50
  finite: 361,892
  missing NaN: 339,092
  +inf: 20
  -inf: 0
  finite quantiles:
0.000     0.000000e+00
0.500     1.499243e+00
0.900     5.918598e+00
0.950     7.336261e+00
0.990     1.500514e+01
0.999     1.111821e+12
1.000    1.468381e+299



In [45]:
# =============================================================================
# Compare PRISM fitted concentrations with tested dose ranges
# =============================================================================

prism_test_ranges = (
    prism_treatments
    .groupby(
        ["broad_id", "screen_id"],
        as_index=False,
    )
    .agg(
        min_test_dose_umol=("dose", "min"),
        max_test_dose_umol=("dose", "max"),
        n_test_doses=("dose", "nunique"),
    )
)

prism_metric_vs_test_range = (
    prism_response_metrics
    .merge(
        prism_test_ranges,
        on=["broad_id", "screen_id"],
        how="left",
        validate="many_to_one",
    )
)

if prism_metric_vs_test_range["max_test_dose_umol"].isna().any():
    raise RuntimeError(
        "Missing tested-dose range after PRISM treatment metadata join."
    )

prism_metric_vs_test_range["ec50_to_max_dose_ratio"] = (
    prism_metric_vs_test_range["ec50"]
    / prism_metric_vs_test_range["max_test_dose_umol"]
)

prism_metric_vs_test_range["ic50_to_max_dose_ratio"] = (
    prism_metric_vs_test_range["ic50"]
    / prism_metric_vs_test_range["max_test_dose_umol"]
)

for metric in ["ec50", "ic50"]:
    ratio = prism_metric_vs_test_range[
        f"{metric}_to_max_dose_ratio"
    ]

    finite_ratio = ratio.loc[np.isfinite(ratio)]

    print(metric.upper())
    print(
        "  finite fitted values:",
        f"{len(finite_ratio):,}",
    )

    for threshold in [1, 10, 100]:
        count = (finite_ratio > threshold).sum()
        print(
            f"  >{threshold}x max tested dose: "
            f"{count:,} "
            f"({count / len(finite_ratio) * 100:.2f}%)"
        )

    print("  ratio quantiles:")
    print(
        finite_ratio
        .quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
        .to_string()
    )
    print()

EC50
  finite fitted values: 701,004
  >1x max tested dose: 31,974 (4.56%)
  >10x max tested dose: 20,321 (2.90%)
  >100x max tested dose: 15,634 (2.23%)
  ratio quantiles:
0.000     1.307191e-07
0.500     4.956066e-02
0.900     5.224792e-01
0.950     8.809106e-01
0.990     2.094209e+07
0.999     7.833772e+74
1.000    3.877887e+303

IC50
  finite fitted values: 361,892
  >1x max tested dose: 5,538 (1.53%)
  >10x max tested dose: 2,012 (0.56%)
  >100x max tested dose: 1,368 (0.38%)
  ratio quantiles:
0.000     0.000000e+00
0.500     1.516869e-01
0.900     5.920123e-01
0.950     7.338840e-01
0.990     1.500456e+00
0.999     1.111821e+11
1.000    1.468381e+298



In [46]:
# =============================================================================
# Validate PRISM dose-response to treatment-metadata linkage
# =============================================================================

prism_treatment_identity = (
    prism_treatments[
        ["broad_id", "screen_id", "name"]
    ]
    .drop_duplicates()
)

treatment_name_counts = (
    prism_treatment_identity
    .groupby(["broad_id", "screen_id"])["name"]
    .nunique(dropna=False)
)

print(
    "broad_id × screen_id pairs with >1 compound name:",
    f"{(treatment_name_counts > 1).sum():,}",
)

dose_response_identity = (
    prism_dose_response_keys[
        ["broad_id", "screen_id", "name"]
    ]
    .drop_duplicates()
)

identity_linkage = (
    dose_response_identity
    .merge(
        prism_treatment_identity,
        on=["broad_id", "screen_id"],
        how="left",
        suffixes=("_dose", "_treatment"),
        validate="many_to_one",
    )
)

missing_treatment_links = (
    identity_linkage["name_treatment"].isna().sum()
)

name_mismatches = (
    identity_linkage["name_treatment"].notna()
    & (
        identity_linkage["name_dose"]
        != identity_linkage["name_treatment"]
    )
).sum()

print(
    "Dose-response broad_id × screen_id pairs:",
    f"{len(dose_response_identity):,}",
)
print(
    "Pairs absent from treatment metadata:",
    f"{missing_treatment_links:,}",
)
print(
    "Compound-name mismatches:",
    f"{name_mismatches:,}",
)

broad_id × screen_id pairs with >1 compound name: 0
Dose-response broad_id × screen_id pairs: 1,618
Pairs absent from treatment metadata: 0
Compound-name mismatches: 0


In [47]:
# =============================================================================
# Audit PRISM response metrics by screen
# =============================================================================

prism_screen_metric_data = (
    prism_metric_vs_test_range
    .assign(
        r2_negative=lambda x: x["r2"] < 0,
        ec50_above_max=lambda x: (
            np.isfinite(x["ec50"])
            & (x["ec50"] > x["max_test_dose_umol"])
        ),
        ic50_finite=lambda x: np.isfinite(x["ic50"]),
        ic50_above_max=lambda x: (
            np.isfinite(x["ic50"])
            & (x["ic50"] > x["max_test_dose_umol"])
        ),
    )
)

prism_screen_metric_summary = (
    prism_screen_metric_data
    .groupby("screen_id")
    .agg(
        response_rows=("row_name", "size"),
        unique_cell_lines=("row_name", "nunique"),
        unique_broad_ids=("broad_id", "nunique"),
        auc_median=("auc", "median"),
        auc_min=("auc", "min"),
        auc_max=("auc", "max"),
        r2_median=("r2", "median"),
        r2_negative_n=("r2_negative", "sum"),
        ec50_above_max_n=("ec50_above_max", "sum"),
        ic50_finite_n=("ic50_finite", "sum"),
        ic50_above_max_n=("ic50_above_max", "sum"),
    )
)

prism_screen_metric_summary["r2_negative_pct"] = (
    prism_screen_metric_summary["r2_negative_n"]
    / prism_screen_metric_summary["response_rows"]
    * 100
)

prism_screen_metric_summary["ec50_above_max_pct"] = (
    prism_screen_metric_summary["ec50_above_max_n"]
    / prism_screen_metric_summary["response_rows"]
    * 100
)

prism_screen_metric_summary["ic50_finite_pct"] = (
    prism_screen_metric_summary["ic50_finite_n"]
    / prism_screen_metric_summary["response_rows"]
    * 100
)

prism_screen_metric_summary["ic50_above_max_pct_of_finite"] = (
    prism_screen_metric_summary["ic50_above_max_n"]
    / prism_screen_metric_summary["ic50_finite_n"]
    * 100
)

prism_screen_metric_summary

,response_rows,unique_cell_lines,unique_broad_ids,auc_median,auc_min,auc_max,r2_median,r2_negative_n,ec50_above_max_n,ic50_finite_n,ic50_above_max_n,r2_negative_pct,ec50_above_max_pct,ic50_finite_pct,ic50_above_max_pct_of_finite
screen_id,,,,,,,,,,,,,,,
HTS002,602495,488,1396,0.916698,0.004174,4.889162,0.232723,143182,24420,323206,5029,23.764845,4.053146,53.644595,1.555974
MTS005,834,451,2,1.090548,0.632434,1.622269,-0.005147,457,123,76,9,54.796163,14.748201,9.112710,11.842105
MTS006,33472,484,73,0.897199,0.066980,2.339337,0.384503,6850,1801,18492,500,20.464866,5.380617,55.246176,2.703872
MTS010,64203,480,147,0.911549,0.199965,1.000000,0.133678,19911,5630,20118,0,31.012570,8.769061,31.334984,0.000000


In [48]:
# =============================================================================
# Audit PRISM compound-name overlap across screens
# =============================================================================

compound_name_screen_presence = (
    prism_treatments[
        ["name", "screen_id"]
    ]
    .drop_duplicates()
    .assign(present=True)
    .pivot(
        index="name",
        columns="screen_id",
        values="present",
    )
    .fillna(False)
    .astype(bool)
)

print(
    "Unique compound names across all screens:",
    f"{len(compound_name_screen_presence):,}",
)

print("\nPairwise compound-name overlap:")
for i, screen_a in enumerate(screen_ids):
    for screen_b in screen_ids[i + 1:]:
        overlap = (
            compound_name_screen_presence[screen_a]
            & compound_name_screen_presence[screen_b]
        ).sum()

        print(
            f"{screen_a} ∩ {screen_b}: "
            f"{overlap:,}"
        )

mts010_names = set(
    prism_treatments.loc[
        prism_treatments["screen_id"] == "MTS010",
        "name",
    ]
)

previous_screen_names = set(
    prism_treatments.loc[
        prism_treatments["screen_id"] != "MTS010",
        "name",
    ]
)

print("\nMTS010 compound-name disposition:")
print(
    "MTS010 names:",
    f"{len(mts010_names):,}",
)
print(
    "MTS010 names present in at least one earlier screen:",
    f"{len(mts010_names & previous_screen_names):,}",
)
print(
    "MTS010 names not present in any earlier screen:",
    f"{len(mts010_names - previous_screen_names):,}",
)

if mts010_names - previous_screen_names:
    print(
        "Names unique to MTS010:",
        sorted(mts010_names - previous_screen_names),
    )

Unique compound names across all screens: 1,448

Pairwise compound-name overlap:
HTS002 ∩ MTS005: 1
HTS002 ∩ MTS006: 20
HTS002 ∩ MTS010: 136
MTS005 ∩ MTS006: 0
MTS005 ∩ MTS010: 0
MTS006 ∩ MTS010: 12

MTS010 compound-name disposition:
MTS010 names: 147
MTS010 names present in at least one earlier screen: 147
MTS010 names not present in any earlier screen: 0


In [49]:
# =============================================================================
# Audit PRISM cell-line mappability by screen
# =============================================================================

prism_screen_cell_mappability = (
    prism_dose_response_ids
    .assign(
        has_depmap_id=lambda x: x["depmap_id"].notna(),
        failed_str=lambda x: ~x["passed_str_profiling"].astype(bool),
    )
    .groupby("screen_id")
    .agg(
        response_rows=("row_name", "size"),
        unique_row_names=("row_name", "nunique"),
        unique_depmap_ids=("depmap_id", "nunique"),
        rows_without_depmap_id=(
            "has_depmap_id",
            lambda x: (~x).sum(),
        ),
        failed_str_rows=("failed_str", "sum"),
    )
)

screen_cell_status = (
    prism_dose_response_ids[
        [
            "screen_id",
            "row_name",
            "depmap_id",
            "passed_str_profiling",
        ]
    ]
    .drop_duplicates()
    .assign(
        has_depmap_id=lambda x: x["depmap_id"].notna(),
        failed_str=lambda x: ~x["passed_str_profiling"].astype(bool),
    )
)

cell_level_summary = (
    screen_cell_status
    .groupby("screen_id")
    .agg(
        cell_rows_without_depmap_id=(
            "has_depmap_id",
            lambda x: (~x).sum(),
        ),
        failed_str_cell_rows=("failed_str", "sum"),
    )
)

prism_screen_cell_mappability = (
    prism_screen_cell_mappability
    .join(cell_level_summary)
)

prism_screen_cell_mappability["mappable_cell_pct"] = (
    prism_screen_cell_mappability["unique_depmap_ids"]
    / prism_screen_cell_mappability["unique_row_names"]
    * 100
)

prism_screen_cell_mappability

,response_rows,unique_row_names,unique_depmap_ids,rows_without_depmap_id,failed_str_rows,cell_rows_without_depmap_id,failed_str_cell_rows,mappable_cell_pct
screen_id,,,,,,,,
HTS002,602495,488,480,9583,9583,8,8,98.360656
MTS005,834,451,444,12,12,7,7,98.447894
MTS006,33472,484,476,542,542,8,8,98.347107
MTS010,64203,480,473,675,675,7,7,98.541667


In [50]:
# =============================================================================
# Validate PRISM unmappable-cell-line explanation
# =============================================================================

unmapped_prism_rows = (
    prism_dose_response_ids["depmap_id"].isna()
)

failed_str_prism_rows = (
    ~prism_dose_response_ids["passed_str_profiling"].astype(bool)
)

if not (
    unmapped_prism_rows
    == failed_str_prism_rows
).all():
    raise RuntimeError(
        "PRISM rows without depmap_id are not fully explained "
        "by failed STR profiling."
    )

unmapped_row_names = sorted(
    prism_dose_response_ids.loc[
        unmapped_prism_rows,
        "row_name",
    ].unique()
)

print(
    "Rows without depmap_id are fully explained by failed STR profiling: PASS"
)
print(
    "Unique failed-STR response row_names:",
    f"{len(unmapped_row_names):,}",
)
print(unmapped_row_names)

Rows without depmap_id are fully explained by failed STR profiling: PASS
Unique failed-STR response row_names: 8
['ACH-000010_FAILED_STR', 'ACH-000028_FAILED_STR', 'ACH-000409_FAILED_STR', 'ACH-000511_FAILED_STR', 'ACH-000539_FAILED_STR', 'ACH-000807_FAILED_STR', 'ACH-000925_FAILED_STR', 'ACH-001078_FAILED_STR']


In [51]:
# =============================================================================
# Define audited acquisition roles and provenance
# =============================================================================

pharmacogenomic_resource_provenance = pd.DataFrame(
    [
        {
            "resource": "ctrp_archive",
            "source": "CTRPv2.0 / CTD²",
            "release": "CTRPv2.0 2015 ExpandedDataset",
            "acquisition_role": "primary_source",
            "phase6_role": "primary CTRP pharmacogenomic resource",
            "notes": (
                "Original provider archive containing post-QC dose-response "
                "curves, compound metadata, cell-line metadata, and "
                "experimental metadata."
            ),
        },
        {
            "resource": "ctrp_pharmacogx",
            "source": "PharmacoGx / ORCESTRA",
            "release": "PSet_CTRPv2",
            "acquisition_role": "secondary_harmonized_resource",
            "phase6_role": "provenance and potential cross-check only",
            "notes": (
                "Retained as a secondary PharmacoSet; not used as the "
                "canonical CTRP response source."
            ),
        },
        {
            "resource": "prism_readme",
            "source": "DepMap PRISM Repurposing",
            "release": "19Q4 Secondary Screen",
            "acquisition_role": "provider_documentation",
            "phase6_role": "supporting documentation",
            "notes": (
                "Provider documentation defining processing, identifiers, "
                "screens, and response metrics."
            ),
        },
        {
            "resource": "prism_cell_line_info",
            "source": "DepMap PRISM Repurposing",
            "release": "19Q4 Secondary Screen",
            "acquisition_role": "primary_metadata",
            "phase6_role": "cell-line identity and lineage metadata",
            "notes": (
                "Provides PRISM row identifiers, DepMap IDs, CCLE names, "
                "tissue annotations, and STR-profiling status."
            ),
        },
        {
            "resource": "prism_dose_response",
            "source": "DepMap PRISM Repurposing",
            "release": "19Q4 Secondary Screen",
            "acquisition_role": "primary_response_resource",
            "phase6_role": "primary PRISM dose-response candidate",
            "notes": (
                "Producer-fitted dose-response curves with AUC, EC50, IC50, "
                "R2, and curve parameters."
            ),
        },
        {
            "resource": "prism_treatment_info",
            "source": "DepMap PRISM Repurposing",
            "release": "19Q4 Secondary Screen",
            "acquisition_role": "primary_metadata",
            "phase6_role": "compound, batch, dose, and screen metadata",
            "notes": (
                "Deterministic metadata link for replicate-collapsed "
                "treatment conditions."
            ),
        },
        {
            "resource": "prism_logfold_change",
            "source": "DepMap PRISM Repurposing",
            "release": "19Q4 Secondary Screen",
            "acquisition_role": "supplementary_response_resource",
            "phase6_role": "potential sensitivity or technical cross-check",
            "notes": (
                "Replicate-collapsed log2-fold-change matrix after "
                "provider processing and ComBat correction."
            ),
        },
    ]
)

pharmacogenomic_resource_provenance

,resource,source,release,acquisition_role,phase6_role,notes
0,ctrp_archive,CTRPv2.0 / CTD²,CTRPv2.0 2015 ExpandedDataset,primary_source,primary CTRP pharmacogenomic resource,Original provider archive containing post-QC d...
1,ctrp_pharmacogx,PharmacoGx / ORCESTRA,PSet_CTRPv2,secondary_harmonized_resource,provenance and potential cross-check only,Retained as a secondary PharmacoSet; not used ...
2,prism_readme,DepMap PRISM Repurposing,19Q4 Secondary Screen,provider_documentation,supporting documentation,"Provider documentation defining processing, id..."
3,prism_cell_line_info,DepMap PRISM Repurposing,19Q4 Secondary Screen,primary_metadata,cell-line identity and lineage metadata,"Provides PRISM row identifiers, DepMap IDs, CC..."
4,prism_dose_response,DepMap PRISM Repurposing,19Q4 Secondary Screen,primary_response_resource,primary PRISM dose-response candidate,"Producer-fitted dose-response curves with AUC,..."
5,prism_treatment_info,DepMap PRISM Repurposing,19Q4 Secondary Screen,primary_metadata,"compound, batch, dose, and screen metadata",Deterministic metadata link for replicate-coll...
6,prism_logfold_change,DepMap PRISM Repurposing,19Q4 Secondary Screen,supplementary_response_resource,potential sensitivity or technical cross-check,Replicate-collapsed log2-fold-change matrix af...


## Acquisition-audit summary and Phase 6 handoff

The acquired CTRP and PRISM resources passed byte-level identity checks and are
suitable for registration as immutable raw inputs. This notebook establishes
resource identity, internal structure, identifier systems, and technical
constraints only; it does not define Phase 6 analytical eligibility rules or
select a pharmacological response metric.

### CTRPv2.0

The canonical CTRP resource is the original CTD²-distributed
`CTRPv2.0_2015_ctd2_ExpandedDataset.zip`.

The archive passed:

- SHA-256 verification against the independently observed local identity;
- ZIP CRC validation across all 16 archive members; and
- validation of all 15 provider-supplied MD5 checksums in `MANIFEST.txt`.

The provider documentation identifies `v20.data.curves_post_qc.txt` as the
post-QC curve-level resource containing AUC sensitivity scores and fitted curve
parameters. Cell-line, compound, and experiment metadata are provided through
the corresponding `v20.meta.*` tables.

The response table contains 395,263 unique
`experiment_id × master_cpd_id` observations, with no orphan experiment or
compound references.

Although `experiment_id` is not row-unique in the experimental metadata,
its mapping to `master_ccl_id` is deterministic: 907 unique experiment IDs map
to 887 cell lines. At the cell-line–compound level, 7,708 of 387,130 pairs
(1.99%) are represented by more than one experiment, with at most three
experiments per pair. No aggregation rule is defined here.

`apparent_ec50_umol` is fully populated but includes strongly extrapolated
curve-fit estimates. Approximately 10.10% of fitted EC50 values exceed the
compound-specific top tested concentration, and 5.32% exceed it by more than
10-fold. This is recorded as a technical property of the resource and is not
used here to select or exclude a response metric.

The separately acquired `PSet_CTRPv2.rds` is retained as a secondary
PharmacoGx/ORCESTRA-derived resource for provenance and possible cross-checking;
it is not treated as the canonical CTRP pharmacological source.

### PRISM Repurposing 19Q4 Secondary Screen

The acquired PRISM resources correspond to the provider-distributed secondary
screen documentation, cell-line metadata, dose-response curves,
replicate-collapsed treatment metadata, and replicate-collapsed log-fold-change
matrix.

The replicate-collapsed treatment metadata contains 13,008 unique treatment
conditions, and these align exactly one-to-one with the 13,008 experimental
columns of the replicate-collapsed log-fold-change matrix.

The provider reports 1,448 compounds. The acquired data contain exactly 1,448
unique compound names but 1,502 `broad_id` values because 54 compound names are
represented by two drug-batch identifiers. Therefore, `broad_id` is retained
as the observational compound-batch identifier and is not equated with a
unique compound entity.

The dose-response table contains 701,004 observations and is unique at
`broad_id × row_name × screen_id`. Collapsing prematurely to compound name
would introduce within-screen ambiguity for 917 observations, arising from two
HTS002 compounds (`doxycycline` and `U-0126`) represented by distinct batches
with non-identical dose grids.

The secondary-screen structure comprises HTS002, MTS005, MTS006, and MTS010.
All 147 compound names present in MTS010 were already represented in an earlier
screen, consistent with the provider description of MTS010 as technical-redo
data. However, screen-specific response properties differ, so no Phase 6 rule
for prioritizing or replacing earlier measurements with MTS010 is defined in
this notebook.

The replicate-collapsed log-fold-change matrix contains 489 response rows,
matching the number reported by the provider. The dose-response table contains
488 unique `row_name` values; `ACH-001192` is present in the log-fold-change
matrix and cell-line metadata but has no dose-response curve.

Eight response identifiers carry a `_FAILED_STR` suffix. All eight lack valid
`depmap_id` and `ccle_name` assignments and are explicitly marked as failing
STR profiling. Across all screens, every dose-response observation lacking a
`depmap_id` is fully explained by failed STR profiling. No identifier repair or
automatic remapping is performed here.

For mapped cell lines, `depmap_id` is stable across screens. One cell line
(`ACH-000037`) has a historical CCLE-name discrepancy across screens while
retaining the same `depmap_id`; therefore, downstream harmonization should use
`depmap_id` as the stable identifier rather than `ccle_name`.

PRISM `auc`, `ec50`, and `ic50` are producer-fitted response metrics. Their
technical properties differ substantially:

- AUC is complete but is not restricted to the interval [0, 1] across all
  screens;
- EC50 is complete but includes strongly extrapolated fitted values;
- IC50 is finite for only approximately 52% of dose-response observations and
  additionally contains non-finite values; and
- R² includes strongly negative values and therefore should not be interpreted
  as a conventional bounded quality metric without additional methodological
  justification.

Approximately 4.56% of finite PRISM EC50 estimates exceed the maximum tested
dose, whereas 1.53% of finite IC50 estimates do so. These values are recorded
for technical characterization only; no response-metric choice or curve-quality
threshold is made here.

### Boundary for downstream analysis

Notebook 109 does not define:

- CTRP or PRISM response-metric priority;
- curve-quality thresholds;
- experiment or batch aggregation rules;
- compound eligibility criteria;
- screen-priority rules;
- cell-line overlap requirements;
- compound-mapping rules across pharmacogenomic resources; or
- any program–drug association or predictive-modeling decision.

Those choices must be made prospectively during the Phase 6 metadata-only
prerequisite audit and subsequently frozen in the Phase 6 analysis contract,
before inspecting consensus-program–drug associations.

In [52]:
# =============================================================================
# Inspect current CTRP and PRISM raw-data registry entries
# =============================================================================

with Paths.raw_data_registry.open("r", encoding="utf-8") as handle:
    raw_data_registry = json.load(handle)

for resource in ["ctrp", "prism"]:
    print(f"{resource.upper()}:")
    print(
        json.dumps(
            raw_data_registry[resource],
            indent=2,
            ensure_ascii=False,
        )
    )
    print()

CTRP:
{
  "source_database": "Cancer Therapeutics Response Portal (CTRP)",
  "provider": "NCI CTD² / Broad Institute",
  "release": "CTRPv2.0 2015 ExpandedDataset",
  "provenance_mode": "file_managed",
  "canonical_dir": "data/raw/ctrp",
  "files": {
    "CTRPv2.0_2015_ctd2_ExpandedDataset.zip": {
      "role": "primary_pharmacogenomic_archive",
      "status": "acquired_and_used",
      "size_bytes": 342737645,
      "sha256": "8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301474b707648689bdee3"
    },
    "PSet_CTRPv2.rds": {
      "role": "secondary_harmonized_pharmacoset",
      "status": "acquired_not_used",
      "source_database": "PharmacoGx / ORCESTRA",
      "doi": "10.5281/zenodo.7826870",
      "size_bytes": 40707609,
      "sha256": "95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade282a5e4e45200ee442e"
    }
  },
  "download_page_url": "https://ctd2-data.nci.nih.gov/Public/Broad/CTRPv2.0_2015_ctd2_ExpandedDataset/"
}

PRISM:
{
  "source_database": "DepMap PRISM Repurposing",
  "p

In [53]:
# =============================================================================
# Define proposed CTRP and PRISM raw-data registry metadata
# =============================================================================

file_identity = (
    pharmacogenomic_file_inventory
    .set_index("resource")[["size_bytes", "sha256"]]
    .to_dict(orient="index")
)


def registry_file(resource, role, status, **metadata):
    return {
        "role": role,
        "status": status,
        **metadata,
        "size_bytes": int(file_identity[resource]["size_bytes"]),
        "sha256": file_identity[resource]["sha256"],
    }


ctrp_registry_entry = {
    "source_database": "Cancer Therapeutics Response Portal (CTRP)",
    "provider": "NCI CTD² / Broad Institute",
    "release": "CTRPv2.0 2015 ExpandedDataset",
    "provenance_mode": "file_managed",
    "canonical_dir": project_relative_path(Paths.ctrp),
    "files": {
        CTRP_ARCHIVE_PATH.name: registry_file(
            "ctrp_archive",
            role="primary_pharmacogenomic_archive",
            status="acquired_and_used",
        ),
        CTRP_PHARMACOGX_PATH.name: registry_file(
            "ctrp_pharmacogx",
            role="secondary_harmonized_pharmacoset",
            status="acquired_not_used",
            source_database="PharmacoGx / ORCESTRA",
            doi="10.5281/zenodo.7826870",
        ),
    },
}

prism_registry_entry = {
    "source_database": "DepMap PRISM Repurposing",
    "provider": "Broad Institute / DepMap",
    "release": "PRISM Repurposing 19Q4 Secondary Screen",
    "provenance_mode": "file_managed",
    "canonical_dir": project_relative_path(Paths.prism),
    "files": {
        PRISM_README_PATH.name: registry_file(
            "prism_readme",
            role="resource_readme",
            status="supporting",
        ),
        PRISM_CELL_LINE_INFO_PATH.name: registry_file(
            "prism_cell_line_info",
            role="cell_line_metadata",
            status="acquired_and_used",
        ),
        PRISM_DOSE_RESPONSE_PATH.name: registry_file(
            "prism_dose_response",
            role="dose_response_curve_parameters",
            status="acquired_and_used",
        ),
        PRISM_TREATMENT_INFO_PATH.name: registry_file(
            "prism_treatment_info",
            role="replicate_collapsed_treatment_metadata",
            status="acquired_and_used",
        ),
        PRISM_LOGFOLD_CHANGE_PATH.name: registry_file(
            "prism_logfold_change",
            role="replicate_collapsed_logfold_change",
            status="supporting",
        ),
    },
}

print("Proposed CTRP registry entry:")
print(json.dumps(ctrp_registry_entry, indent=2, ensure_ascii=False))

print("\nProposed PRISM registry entry:")
print(json.dumps(prism_registry_entry, indent=2, ensure_ascii=False))

Proposed CTRP registry entry:
{
  "source_database": "Cancer Therapeutics Response Portal (CTRP)",
  "provider": "NCI CTD² / Broad Institute",
  "release": "CTRPv2.0 2015 ExpandedDataset",
  "provenance_mode": "file_managed",
  "canonical_dir": "data/raw/ctrp",
  "files": {
    "CTRPv2.0_2015_ctd2_ExpandedDataset.zip": {
      "role": "primary_pharmacogenomic_archive",
      "status": "acquired_and_used",
      "size_bytes": 342737645,
      "sha256": "8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301474b707648689bdee3"
    },
    "PSet_CTRPv2.rds": {
      "role": "secondary_harmonized_pharmacoset",
      "status": "acquired_not_used",
      "source_database": "PharmacoGx / ORCESTRA",
      "doi": "10.5281/zenodo.7826870",
      "size_bytes": 40707609,
      "sha256": "95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade282a5e4e45200ee442e"
    }
  }
}

Proposed PRISM registry entry:
{
  "source_database": "DepMap PRISM Repurposing",
  "provider": "Broad Institute / DepMap",
  "release": "PRISM

In [54]:
# =============================================================================
# Validate proposed CTRP and PRISM registry entries
# =============================================================================

proposed_raw_data_registry = deepcopy(raw_data_registry)

proposed_raw_data_registry["ctrp"] = ctrp_registry_entry
proposed_raw_data_registry["prism"] = prism_registry_entry

validated_raw_data_registry = validate_raw_data_registry(
    proposed_raw_data_registry
)

print("Proposed raw-data registry validation: PASS")

for resource in ["ctrp", "prism"]:
    print(
        f"{resource.upper()}: "
        f"{len(validated_raw_data_registry[resource]['files'])} files"
    )

Proposed raw-data registry validation: PASS
CTRP: 2 files
PRISM: 5 files


In [55]:
# =============================================================================
# Complete source provenance for proposed registry entries
# =============================================================================

ctrp_registry_entry["download_page_url"] = (
    "https://ctd2-data.nci.nih.gov/Public/Broad/"
    "CTRPv2.0_2015_ctd2_ExpandedDataset/"
)

prism_registry_entry["download_page_url"] = (
    "https://depmap.org/portal/data_page/"
    "?release=PRISM+Repurposing+19Q4&tab=allData"
)

proposed_raw_data_registry = deepcopy(raw_data_registry)
proposed_raw_data_registry["ctrp"] = ctrp_registry_entry
proposed_raw_data_registry["prism"] = prism_registry_entry

validated_raw_data_registry = validate_raw_data_registry(
    proposed_raw_data_registry
)

print("Registry provenance completion: PASS")
print(
    "CTRP source:",
    validated_raw_data_registry["ctrp"]["download_page_url"],
)
print(
    "PRISM source:",
    validated_raw_data_registry["prism"]["download_page_url"],
)

Registry provenance completion: PASS
CTRP source: https://ctd2-data.nci.nih.gov/Public/Broad/CTRPv2.0_2015_ctd2_ExpandedDataset/
PRISM source: https://depmap.org/portal/data_page/?release=PRISM+Repurposing+19Q4&tab=allData


In [56]:
# =============================================================================
# Persist audited CTRP and PRISM registry entries
# =============================================================================

with Paths.raw_data_registry.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    json.dump(
        validated_raw_data_registry,
        handle,
        indent=2,
        ensure_ascii=False,
    )
    handle.write("\n")

print(
    "Updated raw-data registry:",
    project_relative_path(Paths.raw_data_registry),
)

Updated raw-data registry: config/raw_data_registry.json


In [57]:
# =============================================================================
# Reload and validate the persisted raw-data registry
# =============================================================================

persisted_raw_data_registry = load_raw_data_registry(
    Paths.raw_data_registry
)

for resource in ["ctrp", "prism"]:
    if (
        persisted_raw_data_registry[resource]
        != validated_raw_data_registry[resource]
    ):
        raise RuntimeError(
            f"Persisted {resource.upper()} registry entry "
            "does not match the validated proposal."
        )

print("Persisted raw-data registry validation: PASS")
print(
    "CTRP registered files:",
    len(persisted_raw_data_registry["ctrp"]["files"]),
)
print(
    "PRISM registered files:",
    len(persisted_raw_data_registry["prism"]["files"]),
)

Persisted raw-data registry validation: PASS
CTRP registered files: 2
PRISM registered files: 5


In [58]:
# =============================================================================
# Summarize registered CTRP and PRISM raw-data handoff
# =============================================================================

registered_handoff_records = []

for dataset_id in ["ctrp", "prism"]:
    dataset_entry = persisted_raw_data_registry[dataset_id]

    for file_name, file_metadata in dataset_entry["files"].items():
        registered_handoff_records.append(
            {
                "dataset": dataset_id,
                "file_name": file_name,
                "role": file_metadata["role"],
                "status": file_metadata["status"],
                "relative_path": (
                    f"{dataset_entry['canonical_dir']}/{file_name}"
                ),
                "size_bytes": file_metadata["size_bytes"],
                "sha256": file_metadata["sha256"],
            }
        )

registered_pharmacogenomic_handoff = pd.DataFrame(
    registered_handoff_records
)

registered_pharmacogenomic_handoff

,dataset,file_name,role,status,relative_path,size_bytes,sha256
0,ctrp,CTRPv2.0_2015_ctd2_ExpandedDataset.zip,primary_pharmacogenomic_archive,acquired_and_used,data/raw/ctrp/CTRPv2.0_2015_ctd2_ExpandedDatas...,342737645,8f62b3b5ed70cfd367cf52ce0a99884dd0a674d1a8c301...
1,ctrp,PSet_CTRPv2.rds,secondary_harmonized_pharmacoset,acquired_not_used,data/raw/ctrp/PSet_CTRPv2.rds,40707609,95c74304c08912f69d5e35e2863c1fe248cd0cbd21ade2...
2,prism,secondary-screen-readme.txt,resource_readme,supporting,data/raw/prism/secondary-screen-readme.txt,7647,ed453e2bbaecff26cf923d8a69290f2c18fd7488d5a7d6...
3,prism,secondary-screen-cell-line-info.csv,cell_line_metadata,acquired_and_used,data/raw/prism/secondary-screen-cell-line-info...,40979,b93436b5f4bcf5fd14589697be4ca8f99ffd99d6392230...
4,prism,secondary-screen-dose-response-curve-parameter...,dose_response_curve_parameters,acquired_and_used,data/raw/prism/secondary-screen-dose-response-...,264340589,88d1013506e0cd6f191a51c5f3fdd3fb2be54f8afb4e19...
5,prism,secondary-screen-replicate-collapsed-treatment...,replicate_collapsed_treatment_metadata,acquired_and_used,data/raw/prism/secondary-screen-replicate-coll...,3788355,9d0d1fb4faa87a63cd84965ec5e2b55a9df5680520a41b...
6,prism,secondary-screen-replicate-collapsed-logfold-c...,replicate_collapsed_logfold_change,supporting,data/raw/prism/secondary-screen-replicate-coll...,91620551,a358beb9efbc96b3d777cbb0212e4cd724080f17970d24...


In [59]:
print(registered_pharmacogenomic_handoff)

  dataset                                          file_name  \
0    ctrp             CTRPv2.0_2015_ctd2_ExpandedDataset.zip   
1    ctrp                                    PSet_CTRPv2.rds   
2   prism                        secondary-screen-readme.txt   
3   prism                secondary-screen-cell-line-info.csv   
4   prism  secondary-screen-dose-response-curve-parameter...   
5   prism  secondary-screen-replicate-collapsed-treatment...   
6   prism  secondary-screen-replicate-collapsed-logfold-c...   

                                     role             status  \
0         primary_pharmacogenomic_archive  acquired_and_used   
1        secondary_harmonized_pharmacoset  acquired_not_used   
2                         resource_readme         supporting   
3                      cell_line_metadata  acquired_and_used   
4          dose_response_curve_parameters  acquired_and_used   
5  replicate_collapsed_treatment_metadata  acquired_and_used   
6      replicate_collapsed_logfold_chan

## Notebook completion

The CTRP and PRISM acquisition audit is complete.

Seven immutable raw resources have been registered with validated byte-level
identity and explicit acquisition roles:

- two CTRP resources, including the canonical CTD² CTRPv2.0 archive; and
- five PRISM Repurposing 19Q4 Secondary Screen resources.

The raw-data registry has been updated and reloaded successfully under the
project validation contract.

No Phase 6 analytical thresholds, response-metric priorities, aggregation
rules, screen-priority rules, compound-eligibility criteria, or
program–drug associations were defined in this notebook.

The next validation step is the independent registry-versus-filesystem audit
implemented in `101_raw_file_audit.ipynb`.